<a href="https://colab.research.google.com/github/LucasMGcode/Simulador-Cache-4-Way/blob/main/notebooks/cache4way_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📖 Manual do Utilizador: Simulador Cache 4-Way

Bem-vindo ao simulador cache `4-way`. Este trabalho foi desenvolvido para ajudar a visualizar como uma cache associativa por conjunto trata leituras, identifica `hit` e `miss`, escolhe uma via e atualiza a política LRU.

O simulador apresenta uma cache associativa por conjunto `4-way` em Verilog, executa um testbench auto-verificável e usa os traces gerados para alimentar visualizações didáticas.

---
### 0. Grupo
Cícero Cipriano Maciel - 102021\
Gabriel Campos Moreira Fernandes - 105461\
Guilherme Nunes Lopes - 105462\
Lucas Moeller Mombach - 105467\
Lucas de Oliveira Mota - 108209

---
### 🧠 1. Como funciona uma cache 4-way?
A memória principal é maior, porém mais lenta que o processador. A cache mantém cópias de blocos usados recentemente para reduzir o tempo médio de acesso.

Neste projeto, cada endereço é dividido em três campos:
- **`tag`:** identifica qual bloco de memória pode estar guardado.
- **`line`:** escolhe o conjunto da cache que deve ser consultado.
- **`block offset`:** escolhe a posição exata dentro do bloco carregado.

Cada conjunto possui **quatro vias**. A cache compara a `tag` procurada com as quatro tags armazenadas em paralelo:
- em um **`hit`**, o dado já está em uma via válida e pode ser devolvido;
- em um **`miss`**, o bloco precisa ser buscado na RAM;
- se o conjunto estiver cheio, a política **LRU** substitui a via menos recentemente usada.

### ⚙️ 2. Entendendo o simulador
O trabalho apresenta três leituras complementares da mesma execução:
- **Datapath:** mostra como o acesso atual percorre o circuito, desde a separação do endereço até a saída `dout`.
- **Log do acesso:** resume os valores relevantes do passo selecionado, como endereço, evento, conjunto, via e estado da FSM.
- **Matriz global da cache:** mostra como todos os conjuntos e vias ficam depois de cada acesso.

Há dois roteiros de execução:
- **Validação técnica:** confirma automaticamente `hit`, `miss`, substituição, indexação por linha e offsets.
- **Demonstração global:** preenche os quatro conjuntos e torna mais visível a evolução da cache inteira.

Ao navegar pelos passos, observe especialmente:
- como a `line` seleciona o conjunto consultado;
- como a via escolhida muda em um `miss`;
- como as idades LRU mudam depois de um `hit`;
- como offsets diferentes acessam dados diferentes dentro do mesmo bloco.

### ▶️ 3. Como usar este notebook
1. Execute as células de instalação, criação dos arquivos e simulação.
2. Leia o datapath estático para reconhecer os blocos do circuito.
3. Use o seletor de roteiro e os botões `Anterior` / `Próximo` para acompanhar os acessos.
4. Compare sempre o datapath local com a matriz global da cache.


In [1]:
#@title 1. Instalar Icarus Verilog
# Instala o Icarus Verilog no ambiente do Colab.
!apt-get update -qq
!apt-get install -y -qq iverilog
!iverilog -V | head -n 3
print("Icarus Verilog instalado com sucesso.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package iverilog.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../iverilog_11.0-1.1_amd64.deb ...
Unpacking iverilog (11.0-1.1) ...
Setting up iverilog (11.0-1.1) ...
Processing triggers for man-db (2.10.2-1) ...
Icarus Verilog version 11.0 (stable) ()

Copyright 1998-2020 Stephen Williams
Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivlpp -V"
Unable to get version from "/usr/lib/x86_64-linux-gnu/ivl/ivl -V -C"/tmp/ivrlh29e09cd2" -C"/usr/lib/x86_64-linux-gnu/ivl/vvp.conf""
Icarus Verilog instalado com sucesso.


## Fontes Verilog

Os módulos estão organizados por função. Cada arquivo fica recolhido por padrão; abra o bloco para revisar o código.


### Circuitos combinacionais e seleção


<details>
<summary><strong><code>comparator.v</code></strong> - Comparador de tags usado em cada via.</summary>

```verilog
module comparator #(
  parameter TAG_BITS = 8
) (
  input  [TAG_BITS-1:0] tag,
  input  [TAG_BITS-1:0] stored_tag,
  output                equal
);
  assign equal = (tag == stored_tag);
endmodule
```
</details>


<details>
<summary><strong><code>encoder4.v</code></strong> - Converte os hits das quatro vias em seleção de via.</summary>

```verilog
module encoder4(
  input  [3:0] in,
  output [1:0] sel,
  output       hit
);
  assign hit = |in;

  assign sel[1] = in[3] | in[2];
  assign sel[0] = in[3] | in[1];
endmodule
```
</details>


<details>
<summary><strong><code>decoder2to4.v</code></strong> - Ativa somente a via escolhida para escrita.</summary>

```verilog
module decoder2to4(
  input  [1:0] sel,
  output [3:0] out
);
  assign out[0] = (sel == 2'd0);
  assign out[1] = (sel == 2'd1);
  assign out[2] = (sel == 2'd2);
  assign out[3] = (sel == 2'd3);
endmodule
```
</details>


<details>
<summary><strong><code>block_counter.v</code></strong> - Percorre os bytes do bloco durante o preenchimento.</summary>

```verilog
module block_counter #(
  parameter BLOCK_BITS = 2,
  parameter BLOCK_SIZE = 4
) (
  input                    clk,
  input                    reset,
  input                    enable,
  output reg [BLOCK_BITS-1:0] out,
  output                   done
);
  assign done = (out == BLOCK_SIZE - 1);

  always @(posedge clk or posedge reset) begin
    if (reset)
      out <= {BLOCK_BITS{1'b0}};
    else if (enable)
      out <= out + 1'b1;
  end
endmodule
```
</details>


### Memórias internas e RAM


<details>
<summary><strong><code>tag_array.v</code></strong> - Tags armazenadas por linha em uma via.</summary>

```verilog
module tag_array #(
  parameter LINE_BITS = 2,
  parameter TAG_BITS = 8,
  parameter LINES = 4
) (
  input                      clk,
  input                      reset,
  input                      wr,
  input  [LINE_BITS-1:0]     line,
  input  [TAG_BITS-1:0]      din,
  output reg [TAG_BITS-1:0]  dout
);
  reg [TAG_BITS-1:0] memory [0:LINES-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < LINES; i = i + 1)
        memory[i] <= {TAG_BITS{1'b0}};
    end else if (wr) begin
      memory[line] <= din;
    end
  end

  always @(*) begin
    dout = memory[line];
  end
endmodule
```
</details>


<details>
<summary><strong><code>valid_array.v</code></strong> - Bits de validade por linha em uma via.</summary>

```verilog
module valid_array #(
  parameter LINE_BITS = 2,
  parameter LINES = 4
) (
  input                  clk,
  input                  reset,
  input                  wr,
  input  [LINE_BITS-1:0] line,
  output reg             dout
);
  reg memory [0:LINES-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < LINES; i = i + 1)
        memory[i] <= 1'b0;
    end else if (wr) begin
      memory[line] <= 1'b1;
    end
  end

  always @(*) begin
    dout = memory[line];
  end
endmodule
```
</details>


<details>
<summary><strong><code>data_array.v</code></strong> - Dados da cache indexados por conjunto e offset.</summary>

```verilog
module data_array #(
  parameter LINE_BITS = 2,
  parameter BLOCK_BITS = 2,
  parameter WAY_SIZE = 16
) (
  input                  clk,
  input                  wr,
  input  [LINE_BITS-1:0] line,
  input  [BLOCK_BITS-1:0] blk,
  input  [7:0]           din,
  output reg [7:0]       dout
);
  reg [7:0] memory [0:WAY_SIZE-1];
  wire [LINE_BITS+BLOCK_BITS-1:0] index;

  assign index = {line, blk};

  always @(posedge clk) begin
    if (wr)
      memory[index] <= din;
  end

  always @(*) begin
    dout = memory[index];
  end
endmodule
```
</details>


<details>
<summary><strong><code>lru_array.v</code></strong> - Idade LRU por linha em uma via.</summary>

```verilog
module lru_array #(
  parameter LINE_BITS = 2,
  parameter LINES = 4,
  parameter [1:0] RESET_AGE = 2'd0
) (
  input                  clk,
  input                  reset,
  input                  wr,
  input  [LINE_BITS-1:0] line,
  input  [1:0]           din,
  output reg [1:0]       dout
);
  reg [1:0] memory [0:LINES-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < LINES; i = i + 1)
        memory[i] <= RESET_AGE;
    end else if (wr) begin
      memory[line] <= din;
    end
  end

  always @(*) begin
    dout = memory[line];
  end
endmodule
```
</details>


<details>
<summary><strong><code>ram.v</code></strong> - Memória principal usada para preencher blocos em caso de miss.</summary>

```verilog
module ram #(
  parameter RAM_BITS = 12,
  parameter RAM_SIZE = 4096
) (
  input                clk,
  input                reset,
  input                wr,
  input  [RAM_BITS-1:0] addr,
  input  [7:0]         din,
  output reg [7:0]     dout
);
  reg [7:0] memory [0:RAM_SIZE-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < RAM_SIZE; i = i + 1)
        memory[i] <= i[7:0];
    end else if (wr) begin
      memory[addr] <= din;
    end
  end

  always @(*) begin
    dout = memory[addr];
  end
endmodule
```
</details>


### Controle e política de substituição


<details>
<summary><strong><code>lru_update.v</code></strong> - Atualização das idades LRU após acesso.</summary>

```verilog
module lru_update(
  input        enable,
  input  [1:0] current_age,
  input  [1:0] accessed_age,
  output reg [1:0] next_age
);
  always @(*) begin
    next_age = current_age;

    if (enable) begin
      if (current_age == accessed_age)
        next_age = 2'd0;
      else if (current_age < accessed_age)
        next_age = current_age + 1'b1;
      else
        next_age = current_age;
    end
  end
endmodule
```
</details>


<details>
<summary><strong><code>cache4_fsm.v</code></strong> - Máquina de estados da leitura: comparação, hit, miss, preenchimento e atualização.</summary>

```verilog
module cache4_fsm(
  input        clk,
  input        reset,
  input        hit,
  input        block_done,
  output reg   fill_active,
  output reg   write_tag_valid,
  output reg   update_lru,
  output reg   done,
  output reg   use_counter_addr,
  output reg   counter_reset,
  output reg [2:0] state_debug
);
  parameter COMPARE    = 3'd0;
  parameter HIT        = 3'd1;
  parameter MISS       = 3'd2;
  parameter FILL_BLOCK = 3'd3;
  parameter UPDATE_TAG = 3'd4;

  reg [2:0] state;
  reg [2:0] next_state;

  always @(posedge clk or posedge reset) begin
    if (reset)
      state <= COMPARE;
    else
      state <= next_state;
  end

  always @(*) begin
    next_state = state;

    case (state)
      COMPARE:
        next_state = hit ? HIT : MISS;

      HIT:
        next_state = COMPARE;

      MISS:
        next_state = FILL_BLOCK;

      FILL_BLOCK:
        next_state = block_done ? UPDATE_TAG : FILL_BLOCK;

      UPDATE_TAG:
        next_state = COMPARE;

      default:
        next_state = COMPARE;
    endcase
  end

  always @(*) begin
    fill_active     = (state == FILL_BLOCK);
    write_tag_valid = (state == UPDATE_TAG);
    update_lru      = (state == HIT) || (state == UPDATE_TAG);
    done            = (state == HIT) || (state == UPDATE_TAG);
    use_counter_addr = (state == FILL_BLOCK);
    counter_reset   = (state != FILL_BLOCK);
    state_debug     = state;
  end
endmodule
```
</details>


### Integração e verificação


<details>
<summary><strong><code>cache4_read_only.v</code></strong> - Módulo principal que conecta vias, comparadores, RAM, FSM e LRU.</summary>

```verilog
`include "encoder4.v"
`include "decoder2to4.v"
`include "comparator.v"
`include "tag_array.v"
`include "valid_array.v"
`include "data_array.v"
`include "ram.v"
`include "block_counter.v"
`include "lru_update.v"
`include "lru_array.v"
`include "cache4_fsm.v"

module cache_4way_read_only #(
  parameter CACHE_SIZE = 64,
  parameter RAM_SIZE = 4096,
  parameter BLOCK_SIZE = 4,
  parameter WAYS = 4,
  parameter CACHE_LINES = CACHE_SIZE / (WAYS * BLOCK_SIZE),
  parameter LINE_BITS = 2,
  parameter RAM_BITS = 12,
  parameter BLOCK_BITS = 2,
  parameter TAG_BITS = RAM_BITS - LINE_BITS - BLOCK_BITS,
  parameter WAY_SIZE = CACHE_SIZE / WAYS
) (
  input                 clk,
  input                 reset,
  input  [RAM_BITS-1:0] address,
  input  [7:0]          din,
  output [7:0]          dout,
  output                done,
  output                hit,
  output [1:0]          selected_way,
  output [2:0]          state_debug
);
  wire [TAG_BITS-1:0] tag;
  wire [LINE_BITS-1:0] line;
  wire [BLOCK_BITS-1:0] blk;

  assign tag = address[RAM_BITS-1:LINE_BITS+BLOCK_BITS];
  assign line = address[LINE_BITS+BLOCK_BITS-1:BLOCK_BITS];
  assign blk = address[BLOCK_BITS-1:0];

  wire [TAG_BITS-1:0] tag0, tag1, tag2, tag3;
  wire v0, v1, v2, v3;
  wire c0, c1, c2, c3;
  wire [3:0] hit_bits;
  wire [1:0] hit_way;

  comparator #(TAG_BITS) cmp0(.tag(tag), .stored_tag(tag0), .equal(c0));
  comparator #(TAG_BITS) cmp1(.tag(tag), .stored_tag(tag1), .equal(c1));
  comparator #(TAG_BITS) cmp2(.tag(tag), .stored_tag(tag2), .equal(c2));
  comparator #(TAG_BITS) cmp3(.tag(tag), .stored_tag(tag3), .equal(c3));

  assign hit_bits = {c3 & v3, c2 & v2, c1 & v1, c0 & v0};
  encoder4 hit_encoder(.in(hit_bits), .sel(hit_way), .hit(hit));

  wire [1:0] lru0, lru1, lru2, lru3;
  wire [1:0] lru_new0, lru_new1, lru_new2, lru_new3;
  wire [1:0] lru_victim;

  assign lru_victim = (lru0 == 2'd3) ? 2'd0 :
                      (lru1 == 2'd3) ? 2'd1 :
                      (lru2 == 2'd3) ? 2'd2 : 2'd3;

  wire [1:0] fill_way;
  assign fill_way = (!v0) ? 2'd0 :
                    (!v1) ? 2'd1 :
                    (!v2) ? 2'd2 :
                    (!v3) ? 2'd3 : lru_victim;

  assign selected_way = hit ? hit_way : fill_way;

  wire fill_active;
  wire write_tag_valid;
  wire update_lru;
  wire use_counter_addr;
  wire counter_reset;
  wire [BLOCK_BITS-1:0] counter_blk;
  wire block_done;
  wire [BLOCK_BITS-1:0] cache_blk;
  wire [RAM_BITS-1:0] ram_addr;
  wire [7:0] ram_dout;

  assign cache_blk = use_counter_addr ? counter_blk : blk;
  assign ram_addr = {tag, line, counter_blk};

  cache4_fsm fsm(
    .clk(clk),
    .reset(reset),
    .hit(hit),
    .block_done(block_done),
    .fill_active(fill_active),
    .write_tag_valid(write_tag_valid),
    .update_lru(update_lru),
    .done(done),
    .use_counter_addr(use_counter_addr),
    .counter_reset(counter_reset),
    .state_debug(state_debug)
  );

  block_counter #(BLOCK_BITS, BLOCK_SIZE) counter(
    .clk(clk),
    .reset(reset | counter_reset),
    .enable(fill_active),
    .out(counter_blk),
    .done(block_done)
  );

  ram #(RAM_BITS, RAM_SIZE) main_memory(
    .clk(clk),
    .reset(reset),
    .wr(1'b0),
    .addr(ram_addr),
    .din(din),
    .dout(ram_dout)
  );

  wire [3:0] write_way;
  decoder2to4 write_decoder(.sel(fill_way), .out(write_way));

  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags0(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[0]), .line(line), .din(tag), .dout(tag0));
  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags1(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[1]), .line(line), .din(tag), .dout(tag1));
  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags2(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[2]), .line(line), .din(tag), .dout(tag2));
  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags3(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[3]), .line(line), .din(tag), .dout(tag3));

  valid_array #(LINE_BITS, CACHE_LINES) valid0(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[0]), .line(line), .dout(v0));
  valid_array #(LINE_BITS, CACHE_LINES) valid1(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[1]), .line(line), .dout(v1));
  valid_array #(LINE_BITS, CACHE_LINES) valid2(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[2]), .line(line), .dout(v2));
  valid_array #(LINE_BITS, CACHE_LINES) valid3(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[3]), .line(line), .dout(v3));

  wire [7:0] data0, data1, data2, data3;

  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way0(.clk(clk), .wr(fill_active & write_way[0]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data0));
  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way1(.clk(clk), .wr(fill_active & write_way[1]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data1));
  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way2(.clk(clk), .wr(fill_active & write_way[2]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data2));
  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way3(.clk(clk), .wr(fill_active & write_way[3]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data3));

  assign dout = (selected_way == 2'd3) ? data3 :
                (selected_way == 2'd2) ? data2 :
                (selected_way == 2'd1) ? data1 : data0;

  wire [1:0] accessed_age;
  assign accessed_age = (selected_way == 2'd3) ? lru3 :
                        (selected_way == 2'd2) ? lru2 :
                        (selected_way == 2'd1) ? lru1 : lru0;

  lru_update lru_update0(.enable(update_lru), .current_age(lru0), .accessed_age(accessed_age), .next_age(lru_new0));
  lru_update lru_update1(.enable(update_lru), .current_age(lru1), .accessed_age(accessed_age), .next_age(lru_new1));
  lru_update lru_update2(.enable(update_lru), .current_age(lru2), .accessed_age(accessed_age), .next_age(lru_new2));
  lru_update lru_update3(.enable(update_lru), .current_age(lru3), .accessed_age(accessed_age), .next_age(lru_new3));

  lru_array #(LINE_BITS, CACHE_LINES, 2'd0) lru_way0(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new0), .dout(lru0));
  lru_array #(LINE_BITS, CACHE_LINES, 2'd1) lru_way1(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new1), .dout(lru1));
  lru_array #(LINE_BITS, CACHE_LINES, 2'd2) lru_way2(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new2), .dout(lru2));
  lru_array #(LINE_BITS, CACHE_LINES, 2'd3) lru_way3(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new3), .dout(lru3));
endmodule
```
</details>


<details>
<summary><strong><code>tb_cache4.v</code></strong> - Testbench auto-verificável que gera os traces das visualizações.</summary>

```verilog
`include "cache4_read_only.v"

module tb_cache4();
  parameter CACHE_SIZE = 64;
  parameter RAM_SIZE = 4096;
  parameter BLOCK_SIZE = 4;
  parameter WAYS = 4;
  parameter CACHE_LINES = CACHE_SIZE / (WAYS * BLOCK_SIZE);
  parameter LINE_BITS = 2;
  parameter RAM_BITS = 12;
  parameter BLOCK_BITS = 2;
  parameter TAG_BITS = RAM_BITS - LINE_BITS - BLOCK_BITS;
  parameter WAY_SIZE = CACHE_SIZE / WAYS;

  reg clk;
  reg reset;
  reg [RAM_BITS-1:0] address;
  reg [7:0] din;
  wire [7:0] dout;
  wire done;
  wire hit;
  wire [1:0] selected_way;
  wire [2:0] state_debug;

  integer failures;
  integer trace_file;
  integer grid_trace_file;
  integer step;

  cache_4way_read_only #(
    CACHE_SIZE,
    RAM_SIZE,
    BLOCK_SIZE,
    WAYS,
    CACHE_LINES,
    LINE_BITS,
    RAM_BITS,
    BLOCK_BITS,
    TAG_BITS,
    WAY_SIZE
  ) Cache (
    .clk(clk),
    .reset(reset),
    .address(address),
    .din(din),
    .dout(dout),
    .done(done),
    .hit(hit),
    .selected_way(selected_way),
    .state_debug(state_debug)
  );

  initial begin
    clk = 0;
    forever #1 clk = ~clk;
  end

  task expect_equal;
    input [127:0] label;
    input integer actual;
    input integer expected;
    begin
      if (actual !== expected) begin
        $display("FAIL %-16s expected=%0d actual=%0d", label, expected, actual);
        failures = failures + 1;
      end else begin
        $display("PASS %-16s value=%0d", label, actual);
      end
    end
  endtask

  task read_and_check;
    input [RAM_BITS-1:0] addr;
    input expected_hit;
    input [1:0] expected_way;
    input [7:0] expected_dout;
    input [1:0] expected_lru0;
    input [1:0] expected_lru1;
    input [1:0] expected_lru2;
    input [1:0] expected_lru3;
    input [255:0] label;
    input [127:0] event_name;
    reg observed_hit;
    reg [1:0] observed_way;
    reg [7:0] observed_dout;
    reg [2:0] observed_state;
    reg [TAG_BITS-1:0] expected_tag;
    reg [LINE_BITS-1:0] expected_line;
    reg [BLOCK_BITS-1:0] expected_blk;
    begin
      step = step + 1;
      expected_tag = addr[RAM_BITS-1:LINE_BITS+BLOCK_BITS];
      expected_line = addr[LINE_BITS+BLOCK_BITS-1:BLOCK_BITS];
      expected_blk = addr[BLOCK_BITS-1:0];

      address = addr;
      @(posedge clk);
      wait(done == 1'b1);
      #1;

      observed_hit = hit;
      observed_way = selected_way;
      observed_dout = dout;
      observed_state = state_debug;

      $display("\nACCESS %-24s addr=%0d tag=%0d line=%0d blk=%0d hit=%0b way=%0d dout=%0d lru={%0d,%0d,%0d,%0d}",
               label,
               address,
               Cache.tag,
               Cache.line,
               Cache.blk,
               observed_hit,
               observed_way,
               observed_dout,
               Cache.lru0,
               Cache.lru1,
               Cache.lru2,
               Cache.lru3);

      expect_equal("hit", observed_hit, expected_hit);
      expect_equal("selected_way", observed_way, expected_way);
      expect_equal("dout", observed_dout, expected_dout);
      expect_equal("tag", Cache.tag, expected_tag);
      expect_equal("line", Cache.line, expected_line);
      expect_equal("blk", Cache.blk, expected_blk);

      @(posedge clk);
      #1;
      expect_equal("lru0", Cache.lru0, expected_lru0);
      expect_equal("lru1", Cache.lru1, expected_lru1);
      expect_equal("lru2", Cache.lru2, expected_lru2);
      expect_equal("lru3", Cache.lru3, expected_lru3);

      $fwrite(trace_file,
              "%0d,%0s,%0s,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d\n",
              step,
              label,
              event_name,
              address,
              Cache.tag,
              Cache.line,
              Cache.blk,
              observed_hit,
              observed_way,
              observed_dout,
              observed_state,
              Cache.lru0,
              Cache.lru1,
              Cache.lru2,
              Cache.lru3,
              Cache.v0,
              Cache.v1,
              Cache.v2,
              Cache.v3,
              Cache.tag0,
              Cache.tag1,
              Cache.tag2,
              Cache.tag3);

      write_grid_snapshot(label, event_name, expected_line, observed_way);
    end
  endtask

  task write_grid_snapshot;
    input [255:0] label;
    input [127:0] event_name;
    input [LINE_BITS-1:0] active_set;
    input [1:0] active_way;
    integer set_idx;
    integer way_idx;
    reg snapshot_valid;
    reg [TAG_BITS-1:0] snapshot_tag;
    reg [1:0] snapshot_lru;
    begin
      for (set_idx = 0; set_idx < CACHE_LINES; set_idx = set_idx + 1) begin
        for (way_idx = 0; way_idx < WAYS; way_idx = way_idx + 1) begin
          case (way_idx)
            0: begin
              snapshot_valid = Cache.valid0.memory[set_idx];
              snapshot_tag = Cache.tags0.memory[set_idx];
              snapshot_lru = Cache.lru_way0.memory[set_idx];
            end
            1: begin
              snapshot_valid = Cache.valid1.memory[set_idx];
              snapshot_tag = Cache.tags1.memory[set_idx];
              snapshot_lru = Cache.lru_way1.memory[set_idx];
            end
            2: begin
              snapshot_valid = Cache.valid2.memory[set_idx];
              snapshot_tag = Cache.tags2.memory[set_idx];
              snapshot_lru = Cache.lru_way2.memory[set_idx];
            end
            default: begin
              snapshot_valid = Cache.valid3.memory[set_idx];
              snapshot_tag = Cache.tags3.memory[set_idx];
              snapshot_lru = Cache.lru_way3.memory[set_idx];
            end
          endcase

          $fwrite(grid_trace_file,
                  "%0d,%0s,%0s,%0d,%0d,%0d,%0d,%0d,%0d,%0d\n",
                  step,
                  label,
                  event_name,
                  active_set,
                  active_way,
                  set_idx,
                  way_idx,
                  snapshot_valid,
                  snapshot_tag,
                  snapshot_lru);
        end
      end
    end
  endtask

  initial begin
    $dumpfile("cache4.vcd");
    $dumpvars(0, tb_cache4);

    failures = 0;
    step = 0;
    trace_file = $fopen("trace.csv", "w");
    if (trace_file == 0) begin
      $display("FAIL could not open trace.csv");
      $fatal(1);
    end
    grid_trace_file = $fopen("trace_grid.csv", "w");
    if (grid_trace_file == 0) begin
      $display("FAIL could not open trace_grid.csv");
      $fatal(1);
    end
    $fwrite(trace_file, "step,label,event,addr,tag,line,blk,hit,selected_way,dout,state,lru0,lru1,lru2,lru3,valid0,valid1,valid2,valid3,tag0,tag1,tag2,tag3\n");
    $fwrite(grid_trace_file, "step,label,event,active_set,active_way,set,way,valid,tag,lru\n");

    din = 8'd0;
    address = 0;
    reset = 1'b1;
    #4;
    reset = 1'b0;

    $display("\n=== VALIDATION TRACE ===");

    read_and_check(12'd0,  1'b0, 2'd0, 8'd0,  2'd0, 2'd1, 2'd2, 2'd3, "miss_fills_invalid_way0", "miss-fill");
    read_and_check(12'd16, 1'b0, 2'd1, 8'd16, 2'd1, 2'd0, 2'd2, 2'd3, "miss_fills_invalid_way1", "miss-fill");
    read_and_check(12'd32, 1'b0, 2'd2, 8'd32, 2'd2, 2'd1, 2'd0, 2'd3, "miss_fills_invalid_way2", "miss-fill");
    read_and_check(12'd48, 1'b0, 2'd3, 8'd48, 2'd3, 2'd2, 2'd1, 2'd0, "miss_fills_invalid_way3", "miss-fill");
    read_and_check(12'd0,  1'b1, 2'd0, 8'd0,  2'd0, 2'd3, 2'd2, 2'd1, "hit_updates_lru", "hit");
    read_and_check(12'd64, 1'b0, 2'd1, 8'd64, 2'd1, 2'd0, 2'd3, 2'd2, "miss_replaces_lru_way1", "miss-replace");

    read_and_check(12'd4,  1'b0, 2'd0, 8'd4,  2'd0, 2'd1, 2'd2, 2'd3, "line1_miss_way0", "miss-line");
    read_and_check(12'd8,  1'b0, 2'd0, 8'd8,  2'd0, 2'd1, 2'd2, 2'd3, "line2_miss_way0", "miss-line");
    read_and_check(12'd12, 1'b0, 2'd0, 8'd12, 2'd0, 2'd1, 2'd2, 2'd3, "line3_miss_way0", "miss-line");
    read_and_check(12'd4,  1'b1, 2'd0, 8'd4,  2'd0, 2'd1, 2'd2, 2'd3, "line1_hit_way0", "hit-line");
    read_and_check(12'd8,  1'b1, 2'd0, 8'd8,  2'd0, 2'd1, 2'd2, 2'd3, "line2_hit_way0", "hit-line");
    read_and_check(12'd12, 1'b1, 2'd0, 8'd12, 2'd0, 2'd1, 2'd2, 2'd3, "line3_hit_way0", "hit-line");

    read_and_check(12'd26, 1'b0, 2'd1, 8'd26, 2'd1, 2'd0, 2'd2, 2'd3, "offset2_miss_way1", "miss-offset");
    read_and_check(12'd24, 1'b1, 2'd1, 8'd24, 2'd1, 2'd0, 2'd2, 2'd3, "offset0_hit_way1", "hit-offset");
    read_and_check(12'd25, 1'b1, 2'd1, 8'd25, 2'd1, 2'd0, 2'd2, 2'd3, "offset1_hit_way1", "hit-offset");
    read_and_check(12'd26, 1'b1, 2'd1, 8'd26, 2'd1, 2'd0, 2'd2, 2'd3, "offset2_hit_way1", "hit-offset");
    read_and_check(12'd27, 1'b1, 2'd1, 8'd27, 2'd1, 2'd0, 2'd2, 2'd3, "offset3_hit_way1", "hit-offset");

    $fclose(trace_file);
    $fclose(grid_trace_file);

    step = 0;
    address = 0;
    reset = 1'b1;
    #4;
    reset = 1'b0;

    trace_file = $fopen("trace_demo.csv", "w");
    if (trace_file == 0) begin
      $display("FAIL could not open trace_demo.csv");
      $fatal(1);
    end
    grid_trace_file = $fopen("trace_grid_demo.csv", "w");
    if (grid_trace_file == 0) begin
      $display("FAIL could not open trace_grid_demo.csv");
      $fatal(1);
    end
    $fwrite(trace_file, "step,label,event,addr,tag,line,blk,hit,selected_way,dout,state,lru0,lru1,lru2,lru3,valid0,valid1,valid2,valid3,tag0,tag1,tag2,tag3\n");
    $fwrite(grid_trace_file, "step,label,event,active_set,active_way,set,way,valid,tag,lru\n");

    $display("\n=== GLOBAL DEMONSTRATION TRACE ===");

    read_and_check(12'd0,  1'b0, 2'd0, 8'd0,  2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set0_way0", "miss-fill");
    read_and_check(12'd16, 1'b0, 2'd1, 8'd16, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set0_way1", "miss-fill");
    read_and_check(12'd32, 1'b0, 2'd2, 8'd32, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set0_way2", "miss-fill");
    read_and_check(12'd48, 1'b0, 2'd3, 8'd48, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set0_way3", "miss-fill");
    read_and_check(12'd4,  1'b0, 2'd0, 8'd4,  2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set1_way0", "miss-fill");
    read_and_check(12'd20, 1'b0, 2'd1, 8'd20, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set1_way1", "miss-fill");
    read_and_check(12'd36, 1'b0, 2'd2, 8'd36, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set1_way2", "miss-fill");
    read_and_check(12'd52, 1'b0, 2'd3, 8'd52, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set1_way3", "miss-fill");
    read_and_check(12'd8,  1'b0, 2'd0, 8'd8,  2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set2_way0", "miss-fill");
    read_and_check(12'd24, 1'b0, 2'd1, 8'd24, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set2_way1", "miss-fill");
    read_and_check(12'd40, 1'b0, 2'd2, 8'd40, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set2_way2", "miss-fill");
    read_and_check(12'd56, 1'b0, 2'd3, 8'd56, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set2_way3", "miss-fill");
    read_and_check(12'd12, 1'b0, 2'd0, 8'd12, 2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set3_way0", "miss-fill");
    read_and_check(12'd28, 1'b0, 2'd1, 8'd28, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set3_way1", "miss-fill");
    read_and_check(12'd44, 1'b0, 2'd2, 8'd44, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set3_way2", "miss-fill");
    read_and_check(12'd60, 1'b0, 2'd3, 8'd60, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set3_way3", "miss-fill");

    read_and_check(12'd0,  1'b1, 2'd0, 8'd0,  2'd0, 2'd3, 2'd2, 2'd1, "phase2_lru_set0_way0", "hit-lru");
    read_and_check(12'd32, 1'b1, 2'd2, 8'd32, 2'd1, 2'd3, 2'd0, 2'd2, "phase2_lru_set0_way2", "hit-lru");
    read_and_check(12'd20, 1'b1, 2'd1, 8'd20, 2'd3, 2'd0, 2'd2, 2'd1, "phase2_lru_set1_way1", "hit-lru");
    read_and_check(12'd52, 1'b1, 2'd3, 8'd52, 2'd3, 2'd1, 2'd2, 2'd0, "phase2_lru_set1_way3", "hit-lru");
    read_and_check(12'd40, 1'b1, 2'd2, 8'd40, 2'd3, 2'd2, 2'd0, 2'd1, "phase2_lru_set2_way2", "hit-lru");
    read_and_check(12'd8,  1'b1, 2'd0, 8'd8,  2'd0, 2'd3, 2'd1, 2'd2, "phase2_lru_set2_way0", "hit-lru");
    read_and_check(12'd60, 1'b1, 2'd3, 8'd60, 2'd3, 2'd2, 2'd1, 2'd0, "phase2_lru_set3_way3", "hit-lru");
    read_and_check(12'd28, 1'b1, 2'd1, 8'd28, 2'd3, 2'd0, 2'd2, 2'd1, "phase2_lru_set3_way1", "hit-lru");

    read_and_check(12'd64, 1'b0, 2'd1, 8'd64, 2'd2, 2'd0, 2'd1, 2'd3, "phase3_replace_set0", "miss-replace");
    read_and_check(12'd68, 1'b0, 2'd0, 8'd68, 2'd0, 2'd2, 2'd3, 2'd1, "phase3_replace_set1", "miss-replace");
    read_and_check(12'd72, 1'b0, 2'd1, 8'd72, 2'd1, 2'd0, 2'd2, 2'd3, "phase3_replace_set2", "miss-replace");
    read_and_check(12'd76, 1'b0, 2'd0, 8'd76, 2'd0, 2'd1, 2'd3, 2'd2, "phase3_replace_set3", "miss-replace");

    read_and_check(12'd1,  1'b1, 2'd0, 8'd1,  2'd0, 2'd1, 2'd2, 2'd3, "phase4_offset_set0_way0", "hit-offset");
    read_and_check(12'd22, 1'b1, 2'd1, 8'd22, 2'd1, 2'd0, 2'd3, 2'd2, "phase4_offset_set1_way1", "hit-offset");
    read_and_check(12'd42, 1'b1, 2'd2, 8'd42, 2'd2, 2'd1, 2'd0, 2'd3, "phase4_offset_set2_way2", "hit-offset");
    read_and_check(12'd63, 1'b1, 2'd3, 8'd63, 2'd1, 2'd2, 2'd3, 2'd0, "phase4_offset_set3_way3", "hit-offset");

    read_and_check(12'd64, 1'b1, 2'd1, 8'd64, 2'd1, 2'd0, 2'd2, 2'd3, "phase5_replaced_hit_set0", "hit-replaced");
    read_and_check(12'd68, 1'b1, 2'd0, 8'd68, 2'd0, 2'd1, 2'd3, 2'd2, "phase5_replaced_hit_set1", "hit-replaced");
    read_and_check(12'd72, 1'b1, 2'd1, 8'd72, 2'd2, 2'd0, 2'd1, 2'd3, "phase5_replaced_hit_set2", "hit-replaced");
    read_and_check(12'd76, 1'b1, 2'd0, 8'd76, 2'd0, 2'd2, 2'd3, 2'd1, "phase5_replaced_hit_set3", "hit-replaced");

    if (failures == 0) begin
      $display("\nALL TESTS PASSED");
      $fclose(trace_file);
      $fclose(grid_trace_file);
      $finish;
    end else begin
      $display("\nTESTS FAILED failures=%0d", failures);
      $fclose(trace_file);
      $fclose(grid_trace_file);
      $fatal(1);
    end
  end
endmodule
```
</details>


### Build

<details>
<summary><strong><code>Makefile</code></strong> - Compila o testbench com Icarus Verilog e executa a simulação.</summary>

```makefile
SIM ?= cache4_sim
TOP ?= tb_cache4.v

.PHONY: sim clean

sim:
	iverilog -g2012 -o $(SIM) $(TOP)
	vvp $(SIM)

clean:
	rm -f $(SIM) *.vcd *.data *.txt *.csv
```
</details>


In [ ]:
#@title 2. Criar arquivos para execução
from pathlib import Path

REPO = Path("/content/Simulador-Cache-4-Way")
SRC_DIR = REPO / "src"
ASSETS_DIR = REPO / "assets"
SRC_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_FILES = {
    'src/comparator.v': r'''module comparator #(
  parameter TAG_BITS = 8
) (
  input  [TAG_BITS-1:0] tag,
  input  [TAG_BITS-1:0] stored_tag,
  output                equal
);
  assign equal = (tag == stored_tag);
endmodule
''',
    'src/encoder4.v': r'''module encoder4(
  input  [3:0] in,
  output [1:0] sel,
  output       hit
);
  assign hit = |in;

  assign sel[1] = in[3] | in[2];
  assign sel[0] = in[3] | in[1];
endmodule
''',
    'src/decoder2to4.v': r'''module decoder2to4(
  input  [1:0] sel,
  output [3:0] out
);
  assign out[0] = (sel == 2'd0);
  assign out[1] = (sel == 2'd1);
  assign out[2] = (sel == 2'd2);
  assign out[3] = (sel == 2'd3);
endmodule
''',
    'src/block_counter.v': r'''module block_counter #(
  parameter BLOCK_BITS = 2,
  parameter BLOCK_SIZE = 4
) (
  input                    clk,
  input                    reset,
  input                    enable,
  output reg [BLOCK_BITS-1:0] out,
  output                   done
);
  assign done = (out == BLOCK_SIZE - 1);

  always @(posedge clk or posedge reset) begin
    if (reset)
      out <= {BLOCK_BITS{1'b0}};
    else if (enable)
      out <= out + 1'b1;
  end
endmodule
''',
    'src/tag_array.v': r'''module tag_array #(
  parameter LINE_BITS = 2,
  parameter TAG_BITS = 8,
  parameter LINES = 4
) (
  input                      clk,
  input                      reset,
  input                      wr,
  input  [LINE_BITS-1:0]     line,
  input  [TAG_BITS-1:0]      din,
  output reg [TAG_BITS-1:0]  dout
);
  reg [TAG_BITS-1:0] memory [0:LINES-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < LINES; i = i + 1)
        memory[i] <= {TAG_BITS{1'b0}};
    end else if (wr) begin
      memory[line] <= din;
    end
  end

  always @(*) begin
    dout = memory[line];
  end
endmodule
''',
    'src/valid_array.v': r'''module valid_array #(
  parameter LINE_BITS = 2,
  parameter LINES = 4
) (
  input                  clk,
  input                  reset,
  input                  wr,
  input  [LINE_BITS-1:0] line,
  output reg             dout
);
  reg memory [0:LINES-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < LINES; i = i + 1)
        memory[i] <= 1'b0;
    end else if (wr) begin
      memory[line] <= 1'b1;
    end
  end

  always @(*) begin
    dout = memory[line];
  end
endmodule
''',
    'src/data_array.v': r'''module data_array #(
  parameter LINE_BITS = 2,
  parameter BLOCK_BITS = 2,
  parameter WAY_SIZE = 16
) (
  input                  clk,
  input                  wr,
  input  [LINE_BITS-1:0] line,
  input  [BLOCK_BITS-1:0] blk,
  input  [7:0]           din,
  output reg [7:0]       dout
);
  reg [7:0] memory [0:WAY_SIZE-1];
  wire [LINE_BITS+BLOCK_BITS-1:0] index;

  assign index = {line, blk};

  always @(posedge clk) begin
    if (wr)
      memory[index] <= din;
  end

  always @(*) begin
    dout = memory[index];
  end
endmodule
''',
    'src/lru_array.v': r'''module lru_array #(
  parameter LINE_BITS = 2,
  parameter LINES = 4,
  parameter [1:0] RESET_AGE = 2'd0
) (
  input                  clk,
  input                  reset,
  input                  wr,
  input  [LINE_BITS-1:0] line,
  input  [1:0]           din,
  output reg [1:0]       dout
);
  reg [1:0] memory [0:LINES-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < LINES; i = i + 1)
        memory[i] <= RESET_AGE;
    end else if (wr) begin
      memory[line] <= din;
    end
  end

  always @(*) begin
    dout = memory[line];
  end
endmodule
''',
    'src/ram.v': r'''module ram #(
  parameter RAM_BITS = 12,
  parameter RAM_SIZE = 4096
) (
  input                clk,
  input                reset,
  input                wr,
  input  [RAM_BITS-1:0] addr,
  input  [7:0]         din,
  output reg [7:0]     dout
);
  reg [7:0] memory [0:RAM_SIZE-1];
  integer i;

  always @(posedge clk or posedge reset) begin
    if (reset) begin
      for (i = 0; i < RAM_SIZE; i = i + 1)
        memory[i] <= i[7:0];
    end else if (wr) begin
      memory[addr] <= din;
    end
  end

  always @(*) begin
    dout = memory[addr];
  end
endmodule
''',
    'src/lru_update.v': r'''module lru_update(
  input        enable,
  input  [1:0] current_age,
  input  [1:0] accessed_age,
  output reg [1:0] next_age
);
  always @(*) begin
    next_age = current_age;

    if (enable) begin
      if (current_age == accessed_age)
        next_age = 2'd0;
      else if (current_age < accessed_age)
        next_age = current_age + 1'b1;
      else
        next_age = current_age;
    end
  end
endmodule
''',
    'src/cache4_fsm.v': r'''module cache4_fsm(
  input        clk,
  input        reset,
  input        hit,
  input        block_done,
  output reg   fill_active,
  output reg   write_tag_valid,
  output reg   update_lru,
  output reg   done,
  output reg   use_counter_addr,
  output reg   counter_reset,
  output reg [2:0] state_debug
);
  parameter COMPARE    = 3'd0;
  parameter HIT        = 3'd1;
  parameter MISS       = 3'd2;
  parameter FILL_BLOCK = 3'd3;
  parameter UPDATE_TAG = 3'd4;

  reg [2:0] state;
  reg [2:0] next_state;

  always @(posedge clk or posedge reset) begin
    if (reset)
      state <= COMPARE;
    else
      state <= next_state;
  end

  always @(*) begin
    next_state = state;

    case (state)
      COMPARE:
        next_state = hit ? HIT : MISS;

      HIT:
        next_state = COMPARE;

      MISS:
        next_state = FILL_BLOCK;

      FILL_BLOCK:
        next_state = block_done ? UPDATE_TAG : FILL_BLOCK;

      UPDATE_TAG:
        next_state = COMPARE;

      default:
        next_state = COMPARE;
    endcase
  end

  always @(*) begin
    fill_active     = (state == FILL_BLOCK);
    write_tag_valid = (state == UPDATE_TAG);
    update_lru      = (state == HIT) || (state == UPDATE_TAG);
    done            = (state == HIT) || (state == UPDATE_TAG);
    use_counter_addr = (state == FILL_BLOCK);
    counter_reset   = (state != FILL_BLOCK);
    state_debug     = state;
  end
endmodule
''',
    'src/cache4_read_only.v': r'''`include "encoder4.v"
`include "decoder2to4.v"
`include "comparator.v"
`include "tag_array.v"
`include "valid_array.v"
`include "data_array.v"
`include "ram.v"
`include "block_counter.v"
`include "lru_update.v"
`include "lru_array.v"
`include "cache4_fsm.v"

module cache_4way_read_only #(
  parameter CACHE_SIZE = 64,
  parameter RAM_SIZE = 4096,
  parameter BLOCK_SIZE = 4,
  parameter WAYS = 4,
  parameter CACHE_LINES = CACHE_SIZE / (WAYS * BLOCK_SIZE),
  parameter LINE_BITS = 2,
  parameter RAM_BITS = 12,
  parameter BLOCK_BITS = 2,
  parameter TAG_BITS = RAM_BITS - LINE_BITS - BLOCK_BITS,
  parameter WAY_SIZE = CACHE_SIZE / WAYS
) (
  input                 clk,
  input                 reset,
  input  [RAM_BITS-1:0] address,
  input  [7:0]          din,
  output [7:0]          dout,
  output                done,
  output                hit,
  output [1:0]          selected_way,
  output [2:0]          state_debug
);
  wire [TAG_BITS-1:0] tag;
  wire [LINE_BITS-1:0] line;
  wire [BLOCK_BITS-1:0] blk;

  assign tag = address[RAM_BITS-1:LINE_BITS+BLOCK_BITS];
  assign line = address[LINE_BITS+BLOCK_BITS-1:BLOCK_BITS];
  assign blk = address[BLOCK_BITS-1:0];

  wire [TAG_BITS-1:0] tag0, tag1, tag2, tag3;
  wire v0, v1, v2, v3;
  wire c0, c1, c2, c3;
  wire [3:0] hit_bits;
  wire [1:0] hit_way;

  comparator #(TAG_BITS) cmp0(.tag(tag), .stored_tag(tag0), .equal(c0));
  comparator #(TAG_BITS) cmp1(.tag(tag), .stored_tag(tag1), .equal(c1));
  comparator #(TAG_BITS) cmp2(.tag(tag), .stored_tag(tag2), .equal(c2));
  comparator #(TAG_BITS) cmp3(.tag(tag), .stored_tag(tag3), .equal(c3));

  assign hit_bits = {c3 & v3, c2 & v2, c1 & v1, c0 & v0};
  encoder4 hit_encoder(.in(hit_bits), .sel(hit_way), .hit(hit));

  wire [1:0] lru0, lru1, lru2, lru3;
  wire [1:0] lru_new0, lru_new1, lru_new2, lru_new3;
  wire [1:0] lru_victim;

  assign lru_victim = (lru0 == 2'd3) ? 2'd0 :
                      (lru1 == 2'd3) ? 2'd1 :
                      (lru2 == 2'd3) ? 2'd2 : 2'd3;

  wire [1:0] fill_way;
  assign fill_way = (!v0) ? 2'd0 :
                    (!v1) ? 2'd1 :
                    (!v2) ? 2'd2 :
                    (!v3) ? 2'd3 : lru_victim;

  assign selected_way = hit ? hit_way : fill_way;

  wire fill_active;
  wire write_tag_valid;
  wire update_lru;
  wire use_counter_addr;
  wire counter_reset;
  wire [BLOCK_BITS-1:0] counter_blk;
  wire block_done;
  wire [BLOCK_BITS-1:0] cache_blk;
  wire [RAM_BITS-1:0] ram_addr;
  wire [7:0] ram_dout;

  assign cache_blk = use_counter_addr ? counter_blk : blk;
  assign ram_addr = {tag, line, counter_blk};

  cache4_fsm fsm(
    .clk(clk),
    .reset(reset),
    .hit(hit),
    .block_done(block_done),
    .fill_active(fill_active),
    .write_tag_valid(write_tag_valid),
    .update_lru(update_lru),
    .done(done),
    .use_counter_addr(use_counter_addr),
    .counter_reset(counter_reset),
    .state_debug(state_debug)
  );

  block_counter #(BLOCK_BITS, BLOCK_SIZE) counter(
    .clk(clk),
    .reset(reset | counter_reset),
    .enable(fill_active),
    .out(counter_blk),
    .done(block_done)
  );

  ram #(RAM_BITS, RAM_SIZE) main_memory(
    .clk(clk),
    .reset(reset),
    .wr(1'b0),
    .addr(ram_addr),
    .din(din),
    .dout(ram_dout)
  );

  wire [3:0] write_way;
  decoder2to4 write_decoder(.sel(fill_way), .out(write_way));

  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags0(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[0]), .line(line), .din(tag), .dout(tag0));
  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags1(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[1]), .line(line), .din(tag), .dout(tag1));
  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags2(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[2]), .line(line), .din(tag), .dout(tag2));
  tag_array #(LINE_BITS, TAG_BITS, CACHE_LINES) tags3(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[3]), .line(line), .din(tag), .dout(tag3));

  valid_array #(LINE_BITS, CACHE_LINES) valid0(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[0]), .line(line), .dout(v0));
  valid_array #(LINE_BITS, CACHE_LINES) valid1(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[1]), .line(line), .dout(v1));
  valid_array #(LINE_BITS, CACHE_LINES) valid2(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[2]), .line(line), .dout(v2));
  valid_array #(LINE_BITS, CACHE_LINES) valid3(.clk(clk), .reset(reset), .wr(write_tag_valid & write_way[3]), .line(line), .dout(v3));

  wire [7:0] data0, data1, data2, data3;

  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way0(.clk(clk), .wr(fill_active & write_way[0]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data0));
  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way1(.clk(clk), .wr(fill_active & write_way[1]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data1));
  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way2(.clk(clk), .wr(fill_active & write_way[2]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data2));
  data_array #(LINE_BITS, BLOCK_BITS, WAY_SIZE) data_way3(.clk(clk), .wr(fill_active & write_way[3]), .line(line), .blk(cache_blk), .din(ram_dout), .dout(data3));

  assign dout = (selected_way == 2'd3) ? data3 :
                (selected_way == 2'd2) ? data2 :
                (selected_way == 2'd1) ? data1 : data0;

  wire [1:0] accessed_age;
  assign accessed_age = (selected_way == 2'd3) ? lru3 :
                        (selected_way == 2'd2) ? lru2 :
                        (selected_way == 2'd1) ? lru1 : lru0;

  lru_update lru_update0(.enable(update_lru), .current_age(lru0), .accessed_age(accessed_age), .next_age(lru_new0));
  lru_update lru_update1(.enable(update_lru), .current_age(lru1), .accessed_age(accessed_age), .next_age(lru_new1));
  lru_update lru_update2(.enable(update_lru), .current_age(lru2), .accessed_age(accessed_age), .next_age(lru_new2));
  lru_update lru_update3(.enable(update_lru), .current_age(lru3), .accessed_age(accessed_age), .next_age(lru_new3));

  lru_array #(LINE_BITS, CACHE_LINES, 2'd0) lru_way0(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new0), .dout(lru0));
  lru_array #(LINE_BITS, CACHE_LINES, 2'd1) lru_way1(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new1), .dout(lru1));
  lru_array #(LINE_BITS, CACHE_LINES, 2'd2) lru_way2(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new2), .dout(lru2));
  lru_array #(LINE_BITS, CACHE_LINES, 2'd3) lru_way3(.clk(clk), .reset(reset), .wr(update_lru), .line(line), .din(lru_new3), .dout(lru3));
endmodule
''',
    'src/tb_cache4.v': r'''`include "cache4_read_only.v"

module tb_cache4();
  parameter CACHE_SIZE = 64;
  parameter RAM_SIZE = 4096;
  parameter BLOCK_SIZE = 4;
  parameter WAYS = 4;
  parameter CACHE_LINES = CACHE_SIZE / (WAYS * BLOCK_SIZE);
  parameter LINE_BITS = 2;
  parameter RAM_BITS = 12;
  parameter BLOCK_BITS = 2;
  parameter TAG_BITS = RAM_BITS - LINE_BITS - BLOCK_BITS;
  parameter WAY_SIZE = CACHE_SIZE / WAYS;

  reg clk;
  reg reset;
  reg [RAM_BITS-1:0] address;
  reg [7:0] din;
  wire [7:0] dout;
  wire done;
  wire hit;
  wire [1:0] selected_way;
  wire [2:0] state_debug;

  integer failures;
  integer trace_file;
  integer grid_trace_file;
  integer step;

  cache_4way_read_only #(
    CACHE_SIZE,
    RAM_SIZE,
    BLOCK_SIZE,
    WAYS,
    CACHE_LINES,
    LINE_BITS,
    RAM_BITS,
    BLOCK_BITS,
    TAG_BITS,
    WAY_SIZE
  ) Cache (
    .clk(clk),
    .reset(reset),
    .address(address),
    .din(din),
    .dout(dout),
    .done(done),
    .hit(hit),
    .selected_way(selected_way),
    .state_debug(state_debug)
  );

  initial begin
    clk = 0;
    forever #1 clk = ~clk;
  end

  task expect_equal;
    input [127:0] label;
    input integer actual;
    input integer expected;
    begin
      if (actual !== expected) begin
        $display("FAIL %-16s expected=%0d actual=%0d", label, expected, actual);
        failures = failures + 1;
      end else begin
        $display("PASS %-16s value=%0d", label, actual);
      end
    end
  endtask

  task read_and_check;
    input [RAM_BITS-1:0] addr;
    input expected_hit;
    input [1:0] expected_way;
    input [7:0] expected_dout;
    input [1:0] expected_lru0;
    input [1:0] expected_lru1;
    input [1:0] expected_lru2;
    input [1:0] expected_lru3;
    input [255:0] label;
    input [127:0] event_name;
    reg observed_hit;
    reg [1:0] observed_way;
    reg [7:0] observed_dout;
    reg [2:0] observed_state;
    reg [TAG_BITS-1:0] expected_tag;
    reg [LINE_BITS-1:0] expected_line;
    reg [BLOCK_BITS-1:0] expected_blk;
    begin
      step = step + 1;
      expected_tag = addr[RAM_BITS-1:LINE_BITS+BLOCK_BITS];
      expected_line = addr[LINE_BITS+BLOCK_BITS-1:BLOCK_BITS];
      expected_blk = addr[BLOCK_BITS-1:0];

      address = addr;
      @(posedge clk);
      wait(done == 1'b1);
      #1;

      observed_hit = hit;
      observed_way = selected_way;
      observed_dout = dout;
      observed_state = state_debug;

      $display("\nACCESS %-24s addr=%0d tag=%0d line=%0d blk=%0d hit=%0b way=%0d dout=%0d lru={%0d,%0d,%0d,%0d}",
               label,
               address,
               Cache.tag,
               Cache.line,
               Cache.blk,
               observed_hit,
               observed_way,
               observed_dout,
               Cache.lru0,
               Cache.lru1,
               Cache.lru2,
               Cache.lru3);

      expect_equal("hit", observed_hit, expected_hit);
      expect_equal("selected_way", observed_way, expected_way);
      expect_equal("dout", observed_dout, expected_dout);
      expect_equal("tag", Cache.tag, expected_tag);
      expect_equal("line", Cache.line, expected_line);
      expect_equal("blk", Cache.blk, expected_blk);

      @(posedge clk);
      #1;
      expect_equal("lru0", Cache.lru0, expected_lru0);
      expect_equal("lru1", Cache.lru1, expected_lru1);
      expect_equal("lru2", Cache.lru2, expected_lru2);
      expect_equal("lru3", Cache.lru3, expected_lru3);

      $fwrite(trace_file,
              "%0d,%0s,%0s,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d,%0d\n",
              step,
              label,
              event_name,
              address,
              Cache.tag,
              Cache.line,
              Cache.blk,
              observed_hit,
              observed_way,
              observed_dout,
              observed_state,
              Cache.lru0,
              Cache.lru1,
              Cache.lru2,
              Cache.lru3,
              Cache.v0,
              Cache.v1,
              Cache.v2,
              Cache.v3,
              Cache.tag0,
              Cache.tag1,
              Cache.tag2,
              Cache.tag3);

      write_grid_snapshot(label, event_name, expected_line, observed_way);
    end
  endtask

  task write_grid_snapshot;
    input [255:0] label;
    input [127:0] event_name;
    input [LINE_BITS-1:0] active_set;
    input [1:0] active_way;
    integer set_idx;
    integer way_idx;
    reg snapshot_valid;
    reg [TAG_BITS-1:0] snapshot_tag;
    reg [1:0] snapshot_lru;
    begin
      for (set_idx = 0; set_idx < CACHE_LINES; set_idx = set_idx + 1) begin
        for (way_idx = 0; way_idx < WAYS; way_idx = way_idx + 1) begin
          case (way_idx)
            0: begin
              snapshot_valid = Cache.valid0.memory[set_idx];
              snapshot_tag = Cache.tags0.memory[set_idx];
              snapshot_lru = Cache.lru_way0.memory[set_idx];
            end
            1: begin
              snapshot_valid = Cache.valid1.memory[set_idx];
              snapshot_tag = Cache.tags1.memory[set_idx];
              snapshot_lru = Cache.lru_way1.memory[set_idx];
            end
            2: begin
              snapshot_valid = Cache.valid2.memory[set_idx];
              snapshot_tag = Cache.tags2.memory[set_idx];
              snapshot_lru = Cache.lru_way2.memory[set_idx];
            end
            default: begin
              snapshot_valid = Cache.valid3.memory[set_idx];
              snapshot_tag = Cache.tags3.memory[set_idx];
              snapshot_lru = Cache.lru_way3.memory[set_idx];
            end
          endcase

          $fwrite(grid_trace_file,
                  "%0d,%0s,%0s,%0d,%0d,%0d,%0d,%0d,%0d,%0d\n",
                  step,
                  label,
                  event_name,
                  active_set,
                  active_way,
                  set_idx,
                  way_idx,
                  snapshot_valid,
                  snapshot_tag,
                  snapshot_lru);
        end
      end
    end
  endtask

  initial begin
    $dumpfile("cache4.vcd");
    $dumpvars(0, tb_cache4);

    failures = 0;
    step = 0;
    trace_file = $fopen("trace.csv", "w");
    if (trace_file == 0) begin
      $display("FAIL could not open trace.csv");
      $fatal(1);
    end
    grid_trace_file = $fopen("trace_grid.csv", "w");
    if (grid_trace_file == 0) begin
      $display("FAIL could not open trace_grid.csv");
      $fatal(1);
    end
    $fwrite(trace_file, "step,label,event,addr,tag,line,blk,hit,selected_way,dout,state,lru0,lru1,lru2,lru3,valid0,valid1,valid2,valid3,tag0,tag1,tag2,tag3\n");
    $fwrite(grid_trace_file, "step,label,event,active_set,active_way,set,way,valid,tag,lru\n");

    din = 8'd0;
    address = 0;
    reset = 1'b1;
    #4;
    reset = 1'b0;

    $display("\n=== VALIDATION TRACE ===");

    read_and_check(12'd0,  1'b0, 2'd0, 8'd0,  2'd0, 2'd1, 2'd2, 2'd3, "miss_fills_invalid_way0", "miss-fill");
    read_and_check(12'd16, 1'b0, 2'd1, 8'd16, 2'd1, 2'd0, 2'd2, 2'd3, "miss_fills_invalid_way1", "miss-fill");
    read_and_check(12'd32, 1'b0, 2'd2, 8'd32, 2'd2, 2'd1, 2'd0, 2'd3, "miss_fills_invalid_way2", "miss-fill");
    read_and_check(12'd48, 1'b0, 2'd3, 8'd48, 2'd3, 2'd2, 2'd1, 2'd0, "miss_fills_invalid_way3", "miss-fill");
    read_and_check(12'd0,  1'b1, 2'd0, 8'd0,  2'd0, 2'd3, 2'd2, 2'd1, "hit_updates_lru", "hit");
    read_and_check(12'd64, 1'b0, 2'd1, 8'd64, 2'd1, 2'd0, 2'd3, 2'd2, "miss_replaces_lru_way1", "miss-replace");

    read_and_check(12'd4,  1'b0, 2'd0, 8'd4,  2'd0, 2'd1, 2'd2, 2'd3, "line1_miss_way0", "miss-line");
    read_and_check(12'd8,  1'b0, 2'd0, 8'd8,  2'd0, 2'd1, 2'd2, 2'd3, "line2_miss_way0", "miss-line");
    read_and_check(12'd12, 1'b0, 2'd0, 8'd12, 2'd0, 2'd1, 2'd2, 2'd3, "line3_miss_way0", "miss-line");
    read_and_check(12'd4,  1'b1, 2'd0, 8'd4,  2'd0, 2'd1, 2'd2, 2'd3, "line1_hit_way0", "hit-line");
    read_and_check(12'd8,  1'b1, 2'd0, 8'd8,  2'd0, 2'd1, 2'd2, 2'd3, "line2_hit_way0", "hit-line");
    read_and_check(12'd12, 1'b1, 2'd0, 8'd12, 2'd0, 2'd1, 2'd2, 2'd3, "line3_hit_way0", "hit-line");

    read_and_check(12'd26, 1'b0, 2'd1, 8'd26, 2'd1, 2'd0, 2'd2, 2'd3, "offset2_miss_way1", "miss-offset");
    read_and_check(12'd24, 1'b1, 2'd1, 8'd24, 2'd1, 2'd0, 2'd2, 2'd3, "offset0_hit_way1", "hit-offset");
    read_and_check(12'd25, 1'b1, 2'd1, 8'd25, 2'd1, 2'd0, 2'd2, 2'd3, "offset1_hit_way1", "hit-offset");
    read_and_check(12'd26, 1'b1, 2'd1, 8'd26, 2'd1, 2'd0, 2'd2, 2'd3, "offset2_hit_way1", "hit-offset");
    read_and_check(12'd27, 1'b1, 2'd1, 8'd27, 2'd1, 2'd0, 2'd2, 2'd3, "offset3_hit_way1", "hit-offset");

    $fclose(trace_file);
    $fclose(grid_trace_file);

    step = 0;
    address = 0;
    reset = 1'b1;
    #4;
    reset = 1'b0;

    trace_file = $fopen("trace_demo.csv", "w");
    if (trace_file == 0) begin
      $display("FAIL could not open trace_demo.csv");
      $fatal(1);
    end
    grid_trace_file = $fopen("trace_grid_demo.csv", "w");
    if (grid_trace_file == 0) begin
      $display("FAIL could not open trace_grid_demo.csv");
      $fatal(1);
    end
    $fwrite(trace_file, "step,label,event,addr,tag,line,blk,hit,selected_way,dout,state,lru0,lru1,lru2,lru3,valid0,valid1,valid2,valid3,tag0,tag1,tag2,tag3\n");
    $fwrite(grid_trace_file, "step,label,event,active_set,active_way,set,way,valid,tag,lru\n");

    $display("\n=== GLOBAL DEMONSTRATION TRACE ===");

    read_and_check(12'd0,  1'b0, 2'd0, 8'd0,  2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set0_way0", "miss-fill");
    read_and_check(12'd16, 1'b0, 2'd1, 8'd16, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set0_way1", "miss-fill");
    read_and_check(12'd32, 1'b0, 2'd2, 8'd32, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set0_way2", "miss-fill");
    read_and_check(12'd48, 1'b0, 2'd3, 8'd48, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set0_way3", "miss-fill");
    read_and_check(12'd4,  1'b0, 2'd0, 8'd4,  2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set1_way0", "miss-fill");
    read_and_check(12'd20, 1'b0, 2'd1, 8'd20, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set1_way1", "miss-fill");
    read_and_check(12'd36, 1'b0, 2'd2, 8'd36, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set1_way2", "miss-fill");
    read_and_check(12'd52, 1'b0, 2'd3, 8'd52, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set1_way3", "miss-fill");
    read_and_check(12'd8,  1'b0, 2'd0, 8'd8,  2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set2_way0", "miss-fill");
    read_and_check(12'd24, 1'b0, 2'd1, 8'd24, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set2_way1", "miss-fill");
    read_and_check(12'd40, 1'b0, 2'd2, 8'd40, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set2_way2", "miss-fill");
    read_and_check(12'd56, 1'b0, 2'd3, 8'd56, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set2_way3", "miss-fill");
    read_and_check(12'd12, 1'b0, 2'd0, 8'd12, 2'd0, 2'd1, 2'd2, 2'd3, "phase1_fill_set3_way0", "miss-fill");
    read_and_check(12'd28, 1'b0, 2'd1, 8'd28, 2'd1, 2'd0, 2'd2, 2'd3, "phase1_fill_set3_way1", "miss-fill");
    read_and_check(12'd44, 1'b0, 2'd2, 8'd44, 2'd2, 2'd1, 2'd0, 2'd3, "phase1_fill_set3_way2", "miss-fill");
    read_and_check(12'd60, 1'b0, 2'd3, 8'd60, 2'd3, 2'd2, 2'd1, 2'd0, "phase1_fill_set3_way3", "miss-fill");

    read_and_check(12'd0,  1'b1, 2'd0, 8'd0,  2'd0, 2'd3, 2'd2, 2'd1, "phase2_lru_set0_way0", "hit-lru");
    read_and_check(12'd32, 1'b1, 2'd2, 8'd32, 2'd1, 2'd3, 2'd0, 2'd2, "phase2_lru_set0_way2", "hit-lru");
    read_and_check(12'd20, 1'b1, 2'd1, 8'd20, 2'd3, 2'd0, 2'd2, 2'd1, "phase2_lru_set1_way1", "hit-lru");
    read_and_check(12'd52, 1'b1, 2'd3, 8'd52, 2'd3, 2'd1, 2'd2, 2'd0, "phase2_lru_set1_way3", "hit-lru");
    read_and_check(12'd40, 1'b1, 2'd2, 8'd40, 2'd3, 2'd2, 2'd0, 2'd1, "phase2_lru_set2_way2", "hit-lru");
    read_and_check(12'd8,  1'b1, 2'd0, 8'd8,  2'd0, 2'd3, 2'd1, 2'd2, "phase2_lru_set2_way0", "hit-lru");
    read_and_check(12'd60, 1'b1, 2'd3, 8'd60, 2'd3, 2'd2, 2'd1, 2'd0, "phase2_lru_set3_way3", "hit-lru");
    read_and_check(12'd28, 1'b1, 2'd1, 8'd28, 2'd3, 2'd0, 2'd2, 2'd1, "phase2_lru_set3_way1", "hit-lru");

    read_and_check(12'd64, 1'b0, 2'd1, 8'd64, 2'd2, 2'd0, 2'd1, 2'd3, "phase3_replace_set0", "miss-replace");
    read_and_check(12'd68, 1'b0, 2'd0, 8'd68, 2'd0, 2'd2, 2'd3, 2'd1, "phase3_replace_set1", "miss-replace");
    read_and_check(12'd72, 1'b0, 2'd1, 8'd72, 2'd1, 2'd0, 2'd2, 2'd3, "phase3_replace_set2", "miss-replace");
    read_and_check(12'd76, 1'b0, 2'd0, 8'd76, 2'd0, 2'd1, 2'd3, 2'd2, "phase3_replace_set3", "miss-replace");

    read_and_check(12'd1,  1'b1, 2'd0, 8'd1,  2'd0, 2'd1, 2'd2, 2'd3, "phase4_offset_set0_way0", "hit-offset");
    read_and_check(12'd22, 1'b1, 2'd1, 8'd22, 2'd1, 2'd0, 2'd3, 2'd2, "phase4_offset_set1_way1", "hit-offset");
    read_and_check(12'd42, 1'b1, 2'd2, 8'd42, 2'd2, 2'd1, 2'd0, 2'd3, "phase4_offset_set2_way2", "hit-offset");
    read_and_check(12'd63, 1'b1, 2'd3, 8'd63, 2'd1, 2'd2, 2'd3, 2'd0, "phase4_offset_set3_way3", "hit-offset");

    read_and_check(12'd64, 1'b1, 2'd1, 8'd64, 2'd1, 2'd0, 2'd2, 2'd3, "phase5_replaced_hit_set0", "hit-replaced");
    read_and_check(12'd68, 1'b1, 2'd0, 8'd68, 2'd0, 2'd1, 2'd3, 2'd2, "phase5_replaced_hit_set1", "hit-replaced");
    read_and_check(12'd72, 1'b1, 2'd1, 8'd72, 2'd2, 2'd0, 2'd1, 2'd3, "phase5_replaced_hit_set2", "hit-replaced");
    read_and_check(12'd76, 1'b1, 2'd0, 8'd76, 2'd0, 2'd2, 2'd3, 2'd1, "phase5_replaced_hit_set3", "hit-replaced");

    if (failures == 0) begin
      $display("\nALL TESTS PASSED");
      $fclose(trace_file);
      $fclose(grid_trace_file);
      $finish;
    end else begin
      $display("\nTESTS FAILED failures=%0d", failures);
      $fclose(trace_file);
      $fclose(grid_trace_file);
      $fatal(1);
    end
  end
endmodule
''',
    'src/Makefile': r'''SIM ?= cache4_sim
TOP ?= tb_cache4.v

.PHONY: sim clean

sim:
	iverilog -g2012 -o $(SIM) $(TOP)
	vvp $(SIM)

clean:
	rm -f $(SIM) *.vcd *.data *.txt *.csv
''',
}

for relative_path, content in PROJECT_FILES.items():
    output_path = REPO / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(content, encoding="utf-8")

DATAPATH_SVG = r'''<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<svg
   width="1600"
   height="900"
   viewBox="0 0 1600 900"
   version="1.1"
   id="cache4way-datapath"
   sodipodi:docname="cache4way_datapath.svg"
   inkscape:version="1.2.2 (b0a8486541, 2022-12-01)"
   xmlns:inkscape="http://www.inkscape.org/namespaces/inkscape"
   xmlns:sodipodi="http://sodipodi.sourceforge.net/DTD/sodipodi-0.dtd"
   xmlns:xlink="http://www.w3.org/1999/xlink"
   xmlns="http://www.w3.org/2000/svg"
   xmlns:svg="http://www.w3.org/2000/svg">
  <sodipodi:namedview
     id="namedview-cache4way"
     pagecolor="#eef3f8"
     bordercolor="#0f172a"
     borderopacity="0.25"
     inkscape:pageopacity="1"
     inkscape:deskcolor="#dbe4ee"
     inkscape:document-units="px"
     inkscape:showpageshadow="2"
     inkscape:pagecheckerboard="0"
     showgrid="false"
     inkscape:zoom="0.868125"
     inkscape:cx="799.42405"
     inkscape:cy="455.0036"
     inkscape:window-width="1920"
     inkscape:window-height="1008"
     inkscape:window-x="1600"
     inkscape:window-y="0"
     inkscape:window-maximized="1"
     inkscape:current-layer="cache4way-datapath" />
  <defs
     id="defs1104">
    <linearGradient
       id="bgGradient"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#f8fbff"
         id="stop1058" />
      <stop
         offset="0.50"
         stop-color="#eef6ff"
         id="stop1060" />
      <stop
         offset="1"
         stop-color="#f7f1e8"
         id="stop1062" />
    </linearGradient>
    <linearGradient
       id="glassPanel"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#ffffff"
         stop-opacity="0.94"
         id="stop1065" />
      <stop
         offset="1"
         stop-color="#eaf3ff"
         stop-opacity="0.76"
         id="stop1067" />
    </linearGradient>
    <linearGradient
       id="wayGlass"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#ffffff"
         stop-opacity="0.98"
         id="stop1070" />
      <stop
         offset="0.56"
         stop-color="#f6fbff"
         stop-opacity="0.90"
         id="stop1072" />
      <stop
         offset="1"
         stop-color="#e7eef8"
         stop-opacity="0.82"
         id="stop1074" />
    </linearGradient>
    <linearGradient
       id="selectedGlass"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#eff6ff"
         id="stop1077" />
      <stop
         offset="0.56"
         stop-color="#dbeafe"
         id="stop1079" />
      <stop
         offset="1"
         stop-color="#bfdbfe"
         id="stop1081" />
    </linearGradient>
    <linearGradient
       id="signalGlass"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#ecfeff"
         stop-opacity="0.98"
         id="stop1084" />
      <stop
         offset="1"
         stop-color="#dbeafe"
         stop-opacity="0.92"
         id="stop1086" />
    </linearGradient>
    <linearGradient
       id="controlGlass"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#fff7ed"
         stop-opacity="0.98"
         id="stop1089" />
      <stop
         offset="1"
         stop-color="#fef3c7"
         stop-opacity="0.90"
         id="stop1091" />
    </linearGradient>
    <linearGradient
       id="memoryGlass"
       x1="0"
       y1="0"
       x2="1"
       y2="1">
      <stop
         offset="0"
         stop-color="#ecfdf5"
         stop-opacity="0.98"
         id="stop1094" />
      <stop
         offset="1"
         stop-color="#d1fae5"
         stop-opacity="0.90"
         id="stop1096" />
    </linearGradient>
    <marker
       id="arrow"
       viewBox="0 0 10 10"
       refX="9"
       refY="5"
       markerWidth="9"
       markerHeight="9"
       orient="auto-start-reverse">
      <path
         d="M 0 0 L 10 5 L 0 10 z"
         fill="#334155"
         id="path1099" />
    </marker>
    <marker
       id="arrowBlue"
       viewBox="0 0 10 10"
       refX="9"
       refY="5"
       markerWidth="9"
       markerHeight="9"
       orient="auto-start-reverse">
      <path
         d="M 0 0 L 10 5 L 0 10 z"
         fill="#2563eb"
         id="path1099b" />
    </marker>
    <style
       id="style1102"><![CDATA[
      .title { font: 800 30px Georgia, serif; fill: #172033; letter-spacing: 0.2px; }
      .subtitle { font: 500 14px Georgia, serif; fill: #64748b; }
      .zone-title { font: 800 15px Georgia, serif; fill: #1e293b; }
      .label { font: 700 13px Georgia, serif; fill: #334155; }
      .small { font: 500 12px Georgia, serif; fill: #64748b; }
      .hint { font: 500 11px Georgia, serif; fill: #64748b; }
      .value { font: 800 16px ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; fill: #0f172a; }
      .mid-value { font: 800 14px ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; fill: #0f172a; }
      .tiny-value { font: 800 12px ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; fill: #0f172a; }
      .panel { fill: url(#glassPanel); stroke: #b7c6d8; stroke-opacity: 0.95; stroke-width: 2; }
      .shadow { fill: #cbd5e1; opacity: 0.28; }
      .way-shadow { fill: #94a3b8; opacity: 0.18; }
      .signal { fill: url(#signalGlass); stroke: #0284c7; stroke-width: 1.5; }
      .logic { fill: #eef2ff; stroke: #4f46e5; stroke-width: 1.5; }
      .control { fill: url(#controlGlass); stroke: #f59e0b; stroke-width: 1.5; }
      .memory { fill: url(#memoryGlass); stroke: #10b981; stroke-width: 1.5; }
      .way-card { fill: url(#wayGlass); stroke: #8aa0b8; stroke-width: 2.2; }
      .way-card.selected { fill: url(#selectedGlass); stroke: #2563eb; stroke-width: 4.5; }
      .event-badge { fill: #fef3c7; stroke: #f59e0b; stroke-width: 2.4; }
      .event-badge.hit { fill: #dcfce7; stroke: #16a34a; }
      .event-badge.miss { fill: #ffedd5; stroke: #ea580c; }
      .event-badge.replace { fill: #fee2e2; stroke: #dc2626; }
      .chip { fill: #f8fafc; fill-opacity: 0.94; stroke: #cbd5e1; stroke-width: 1.1; }
      .chip-hot { fill: #eff6ff; stroke: #38bdf8; stroke-width: 1.2; }
      .data-chip { fill: #ecfdf5; stroke: #10b981; stroke-width: 1.2; }
      .lru-chip { fill: #fff7ed; stroke: #fb923c; stroke-width: 1.2; }
      .bus { stroke: #334155; stroke-width: 2.4; fill: none; marker-end: url(#arrow); stroke-linecap: round; stroke-linejoin: round; }
      .data-bus { stroke: #94a3b8; stroke-width: 2.2; fill: none; marker-end: url(#arrow); stroke-linecap: round; stroke-linejoin: round; opacity: 0.65; }
      .data-bus.selected { stroke: #2563eb; stroke-width: 4; marker-end: url(#arrowBlue); opacity: 1; }
      .fill-bus { stroke: #059669; stroke-width: 2.4; fill: none; marker-end: url(#arrow); stroke-linecap: round; stroke-linejoin: round; stroke-dasharray: 8 6; }
      .fill-bus-tail { stroke: #059669; stroke-width: 2.4; fill: none; marker-end: none; stroke-linecap: round; stroke-linejoin: round; stroke-dasharray: 8 6; }
      .fill-arrow { fill: #059669; stroke: #047857; stroke-width: 1; }
      .control-bus { stroke: #d97706; stroke-width: 2.2; fill: none; marker-end: url(#arrow); stroke-linecap: round; stroke-linejoin: round; }
      .soft-bus { stroke: #94a3b8; stroke-width: 1.5; fill: none; marker-end: url(#arrow); stroke-linecap: round; stroke-linejoin: round; }
    ]]></style>
    <linearGradient
       inkscape:collect="always"
       xlink:href="#glassPanel"
       id="linearGradient1434"
       x1="63.268067"
       y1="608.09508"
       x2="295.51793"
       y2="840.34494"
       gradientTransform="matrix(1.2482456,0,0,0.87859956,0.07348176,-47.321895)"
       gradientUnits="userSpaceOnUse" />
  </defs>
  <rect
     x="0"
     y="0"
     width="1600"
     height="900"
     fill="url(#bgGradient)"
     id="rect1106" />
  <circle
     cx="1320"
     cy="96"
     r="184"
     fill="#bfdbfe"
     opacity="0.20"
     id="circle1108" />
  <circle
     cx="188"
     cy="768"
     r="216"
     fill="#fed7aa"
     opacity="0.20"
     id="circle1110" />
  <circle
     cx="760"
     cy="-40"
     r="232"
     fill="#dcfce7"
     opacity="0.18"
     id="circle1112" />
  <text
     x="48"
     y="52"
     class="title"
     id="text1114">Cache 4-Way Read-Only Datapath</text>
  <text
     x="48"
     y="78"
     class="subtitle"
     id="text1116">Grid 16:9 com espacamento de 8px: endereco, lookup paralelo, decisao de hit/miss e preenchimento de bloco</text>
  <rect
     id="event-badge"
     class="event-badge"
     x="1240"
     y="28"
     width="312"
     height="64"
     rx="16" />
  <text
     x="1264"
     y="54"
     class="label"
     id="text1119">Evento do acesso</text>
  <text
     x="1264"
     y="80"
     class="value"
     id="text1121">@event</text>
  <!-- Column 1: address and summary -->
  <rect
     class="shadow"
     x="56"
     y="136"
     width="360"
     height="596"
     rx="28"
     id="rect1123" />
  <rect
     class="panel"
     x="48"
     y="128"
     width="360"
     height="596"
     rx="28"
     id="rect1125" />
  <text
     x="80"
     y="160"
     class="zone-title"
     id="text1127">1. Entrada e campos</text>
  <text
     x="80"
     y="184"
     class="small"
     id="text1129">Endereco de 12 bits antes do lookup.</text>
  <rect
     class="signal"
     x="80"
     y="208"
     width="296"
     height="80"
     rx="16"
     id="rect1131" />
  <text
     x="104"
     y="232"
     class="label"
     id="text1133">Endereco atual</text>
  <text
     x="104"
     y="256"
     class="mid-value"
     id="text1135">decimal @addr</text>
  <text
     x="104"
     y="278"
     class="tiny-value"
     id="text1136">bin @addr_bin</text>
  <text
     x="80"
     y="312"
     class="label"
     id="text1136a">Separacao binaria</text>
  <rect
     class="chip-hot"
     x="80"
     y="328"
     width="128"
     height="64"
     rx="14"
     id="rect1137" />
  <text
     x="104"
     y="350"
     class="small"
     id="text1139">tag [11:4]</text>
  <text
     x="96"
     y="374"
     class="tiny-value"
     id="text1141">@tag_bin</text>
  <rect
     class="chip-hot"
     x="224"
     y="328"
     width="64"
     height="64"
     rx="14"
     id="rect1143" />
  <text
     x="240"
     y="350"
     class="small"
     id="text1145">line</text>
  <text
     x="236"
     y="374"
     class="tiny-value"
     id="text1147">@line_bin</text>
  <rect
     class="chip-hot"
     x="304"
     y="328"
     width="72"
     height="64"
     rx="14"
     id="rect1149" />
  <text
     x="324"
     y="350"
     class="small"
     id="text1151">blk</text>
  <text
     x="316"
     y="374"
     class="tiny-value"
     id="text1153">@blk_bin</text>
  <text
     x="80"
     y="420"
     class="hint"
     id="text1155">line escolhe o conjunto consultado.</text>
  <text
     x="80"
     y="438"
     class="hint"
     id="text1157">tag compara nas 4 vias; blk escolhe o byte.</text>
  <rect
     class="shadow"
     x="88"
     y="496"
     width="288"
     height="184"
     rx="24"
     id="rect1159" />
  <rect
     class="panel"
     x="80.047241"
     y="488.04724"
     width="287.90552"
     height="201.8605"
     rx="23.992126"
     id="rect1161"
     style="fill:url(#linearGradient1434)" />
  <text
     x="104"
     y="520"
     class="zone-title"
     id="text1163">Resumo do passo</text>
  <text
     x="104"
     y="548"
     class="tiny-value"
     id="text1165">event=@event</text>
  <text
     x="104"
     y="572"
     class="tiny-value"
     id="text1167">addr=@addr | tag=@tag</text>
  <text
     x="104"
     y="596"
     class="tiny-value"
     id="text1169">line=@line | blk=@blk</text>
  <text
     x="104"
     y="620"
     class="tiny-value"
     id="text1171">hit=@hit | way=@selected_way</text>
  <text
     x="104"
     y="644"
     class="tiny-value"
     id="text1173">dout=@dout</text>
  <text
     x="104"
     y="668"
     class="tiny-value"
     id="text1175">state=@state</text>
  <!-- Column 2: cache set -->
  <rect
     class="shadow"
     x="448"
     y="136"
     width="720"
     height="596"
     rx="28"
     id="rect1177" />
  <rect
     class="panel"
     x="440"
     y="128"
     width="720"
     height="596"
     rx="28"
     id="rect1179" />
  <text
     x="472"
     y="160"
     class="zone-title"
     id="text1181">2. Conjunto selecionado pela line @line</text>
  <text
     x="472"
     y="184"
     class="small"
     id="text1183">Quatro vias consultadas em paralelo; comparador = tag armazenada + valid.</text>
  <g
     id="way0-group">
    <rect
       class="way-shadow"
       x="480"
       y="216"
       width="656"
       height="104"
       rx="24"
       id="rect1185" />
    <rect
       id="way0-card"
       class="way-card"
       x="472"
       y="208"
       width="656"
       height="104"
       rx="24" />
    <text
       x="496"
       y="236"
       class="label"
       id="text1188">Way 0</text>
    <rect
       class="logic"
       x="496"
       y="256"
       width="96"
       height="40"
       rx="14"
       id="rect1190" />
    <text
       x="520"
       y="281"
       class="tiny-value"
       id="text1192">cmp0</text>
    <rect
       class="chip"
       x="616"
       y="256"
       width="96"
       height="40"
       rx="14"
       id="rect1194" />
    <text
       x="632"
       y="281"
       class="tiny-value"
       id="text1196">V=@valid0</text>
    <rect
       class="chip"
       x="736"
       y="256"
       width="112"
       height="40"
       rx="14"
       id="rect1198" />
    <text
       x="754"
       y="281"
       class="tiny-value"
       id="text1200">Tag=@tag0</text>
    <rect
       class="data-chip"
       x="872"
       y="256"
       width="96"
       height="40"
       rx="14"
       id="rect1202" />
    <text
       x="894"
       y="281"
       class="tiny-value"
       id="text1204">Data0</text>
    <rect
       class="lru-chip"
       x="992"
       y="256"
       width="112"
       height="40"
       rx="14"
       id="rect1206" />
    <text
       x="1010"
       y="281"
       class="tiny-value"
       id="text1208">LRU=@lru0</text>
  </g>
  <g
     id="way1-group">
    <rect
       class="way-shadow"
       x="480"
       y="344"
       width="656"
       height="104"
       rx="24"
       id="rect1211" />
    <rect
       id="way1-card"
       class="way-card"
       x="472"
       y="336"
       width="656"
       height="104"
       rx="24" />
    <text
       x="496"
       y="364"
       class="label"
       id="text1214">Way 1</text>
    <rect
       class="logic"
       x="496"
       y="384"
       width="96"
       height="40"
       rx="14"
       id="rect1216" />
    <text
       x="520"
       y="409"
       class="tiny-value"
       id="text1218">cmp1</text>
    <rect
       class="chip"
       x="616"
       y="384"
       width="96"
       height="40"
       rx="14"
       id="rect1220" />
    <text
       x="632"
       y="409"
       class="tiny-value"
       id="text1222">V=@valid1</text>
    <rect
       class="chip"
       x="736"
       y="384"
       width="112"
       height="40"
       rx="14"
       id="rect1224" />
    <text
       x="754"
       y="409"
       class="tiny-value"
       id="text1226">Tag=@tag1</text>
    <rect
       class="data-chip"
       x="872"
       y="384"
       width="96"
       height="40"
       rx="14"
       id="rect1228" />
    <text
       x="894"
       y="409"
       class="tiny-value"
       id="text1230">Data1</text>
    <rect
       class="lru-chip"
       x="992"
       y="384"
       width="112"
       height="40"
       rx="14"
       id="rect1232" />
    <text
       x="1010"
       y="409"
       class="tiny-value"
       id="text1234">LRU=@lru1</text>
  </g>
  <g
     id="way2-group">
    <rect
       class="way-shadow"
       x="480"
       y="472"
       width="656"
       height="104"
       rx="24"
       id="rect1237" />
    <rect
       id="way2-card"
       class="way-card"
       x="472"
       y="464"
       width="656"
       height="104"
       rx="24" />
    <text
       x="496"
       y="492"
       class="label"
       id="text1240">Way 2</text>
    <rect
       class="logic"
       x="496"
       y="512"
       width="96"
       height="40"
       rx="14"
       id="rect1242" />
    <text
       x="520"
       y="537"
       class="tiny-value"
       id="text1244">cmp2</text>
    <rect
       class="chip"
       x="616"
       y="512"
       width="96"
       height="40"
       rx="14"
       id="rect1246" />
    <text
       x="632"
       y="537"
       class="tiny-value"
       id="text1248">V=@valid2</text>
    <rect
       class="chip"
       x="736"
       y="512"
       width="112"
       height="40"
       rx="14"
       id="rect1250" />
    <text
       x="754"
       y="537"
       class="tiny-value"
       id="text1252">Tag=@tag2</text>
    <rect
       class="data-chip"
       x="872"
       y="512"
       width="96"
       height="40"
       rx="14"
       id="rect1254" />
    <text
       x="894"
       y="537"
       class="tiny-value"
       id="text1256">Data2</text>
    <rect
       class="lru-chip"
       x="992"
       y="512"
       width="112"
       height="40"
       rx="14"
       id="rect1258" />
    <text
       x="1010"
       y="537"
       class="tiny-value"
       id="text1260">LRU=@lru2</text>
  </g>
  <g
     id="way3-group">
    <rect
       class="way-shadow"
       x="480"
       y="600"
       width="656"
       height="104"
       rx="24"
       id="rect1263" />
    <rect
       id="way3-card"
       class="way-card"
       x="472"
       y="592"
       width="656"
       height="104"
       rx="24" />
    <text
       x="496"
       y="620"
       class="label"
       id="text1266">Way 3</text>
    <rect
       class="logic"
       x="496"
       y="640"
       width="96"
       height="40"
       rx="14"
       id="rect1268" />
    <text
       x="520"
       y="665"
       class="tiny-value"
       id="text1270">cmp3</text>
    <rect
       class="chip"
       x="616"
       y="640"
       width="96"
       height="40"
       rx="14"
       id="rect1272" />
    <text
       x="632"
       y="665"
       class="tiny-value"
       id="text1274">V=@valid3</text>
    <rect
       class="chip"
       x="736"
       y="640"
       width="112"
       height="40"
       rx="14"
       id="rect1276" />
    <text
       x="754"
       y="665"
       class="tiny-value"
       id="text1278">Tag=@tag3</text>
    <rect
       class="data-chip"
       x="872"
       y="640"
       width="96"
       height="40"
       rx="14"
       id="rect1280" />
    <text
       x="894"
       y="665"
       class="tiny-value"
       id="text1282">Data3</text>
    <rect
       class="lru-chip"
       x="992"
       y="640"
       width="112"
       height="40"
       rx="14"
       id="rect1284" />
    <text
       x="1010"
       y="665"
       class="tiny-value"
       id="text1286">LRU=@lru3</text>
  </g>
  <!-- Column 3: decision and output -->
  <rect
     class="shadow"
     x="1200"
     y="136"
     width="360"
     height="596"
     rx="28"
     id="rect1289" />
  <rect
     class="panel"
     x="1192"
     y="128"
     width="360"
     height="596"
     rx="28"
     id="rect1291" />
  <text
     x="1224"
     y="160"
     class="zone-title"
     id="text1293">3. Decisao e saida</text>
  <text
     x="1224"
     y="184"
     class="small"
     id="text1295">Hit, via selecionada e dado final.</text>
  <rect
     class="logic"
     x="1224"
     y="224"
     width="296"
     height="88"
     rx="16"
     id="rect1297" />
  <text
     x="1256"
     y="252"
     class="label"
     id="text1299">Hit encoder</text>
  <text
     x="1256"
     y="276"
     class="small"
     id="text1301">hit bits das 4 vias</text>
  <text
     x="1256"
     y="300"
     class="value"
     id="text1303">hit = @hit</text>
  <rect
     class="logic"
     x="1224"
     y="376"
     width="296"
     height="112"
     rx="16"
     id="rect1305" />
  <text
     x="1256"
     y="404"
     class="label"
     id="text1307">Mux 4:1</text>
  <text
     x="1256"
     y="428"
     class="small"
     id="text1309">seleciona data way</text>
  <text
     x="1256"
     y="452"
     class="small"
     id="text1311">selected_way</text>
  <text
     x="1256"
     y="476"
     class="value"
     id="text1313">@selected_way</text>
  <rect
     class="signal"
     x="1224"
     y="568"
     width="296"
     height="88"
     rx="16"
     id="rect1315" />
  <text
     x="1256"
     y="600"
     class="label"
     id="text1317">Saida da cache</text>
  <text
     x="1256"
     y="632"
     class="value"
     id="text1319">dout = @dout</text>
  <!-- Structured lookup and datapath routes -->
  <path
     class="bus"
     d="M 256 328 C 256 296, 400 296, 440 240"
     id="path1321" />
  <path
     class="soft-bus"
     d="M 408 360 C 424 360, 424 260, 472 260"
     id="path1323" />
  <path
     class="soft-bus"
     d="M 408 360 C 424 360, 424 388, 472 388"
     id="path1325" />
  <path
     class="soft-bus"
     d="M 408 360 C 424 360, 424 516, 472 516"
     id="path1327" />
  <path
     class="soft-bus"
     d="M 408 360 C 424 360, 424 644, 472 644"
     id="path1329" />
  <path
     class="data-bus"
     d="M 1128 276 C 1160 276, 1160 268, 1224 268"
     id="way0-data-path" />
  <path
     class="data-bus"
     d="M 1128 404 C 1160 404, 1160 416, 1224 416"
     id="way1-data-path" />
  <path
     class="data-bus"
     d="M 1128 532 C 1160 532, 1160 440, 1224 440"
     id="way2-data-path" />
  <path
     class="data-bus"
     d="M 1128 660 C 1160 660, 1160 464, 1224 464"
     id="way3-data-path" />
  <path
     class="bus"
     d="M 1372 312 C 1372 344, 1372 344, 1372 376"
     id="path1339" />
  <path
     class="bus"
     d="M 1372 488 C 1372 528, 1372 528, 1372 568"
     id="path1341" />
  <!-- Bottom control and fill path -->
  <rect
     class="shadow"
     x="56"
     y="756"
     width="1504"
     height="104"
     rx="28"
     id="rect1343" />
  <rect
     class="panel"
     x="48"
     y="748"
     width="1504"
     height="104"
     rx="28"
     id="rect1345" />
  <text
     x="80"
     y="780"
     class="zone-title"
     id="text1347">Controle e preenchimento em miss</text>
  <rect
     class="control"
     x="80"
     y="796"
     width="200"
     height="48"
     rx="16"
     id="rect1349" />
  <text
     x="104"
     y="826"
     class="mid-value"
     id="text1351">FSM @state</text>
  <rect
     class="control"
     x="320"
     y="796"
     width="240"
     height="48"
     rx="16"
     id="rect1353" />
  <text
     x="344"
     y="826"
     class="mid-value"
     id="text1355">Decoder way @selected_way</text>
  <rect
     class="control"
     x="600"
     y="796"
     width="300"
     height="48"
     rx="16"
     id="rect1357" />
  <text
     x="624"
     y="826"
     class="mid-value"
     id="text1359">LRU @lru0 @lru1 @lru2 @lru3</text>
  <rect
     class="memory"
     x="940"
     y="796"
     width="220"
     height="48"
     rx="16"
     id="rect1361" />
  <text
     x="964"
     y="826"
     class="mid-value"
     id="text1363">RAM addr @addr</text>
  <rect
     class="memory"
     x="1200"
     y="796"
     width="220"
     height="48"
     rx="16"
     id="rect1365" />
  <text
     x="1224"
     y="826"
     class="mid-value"
     id="text1367">Counter blk @blk</text>
  <path
     class="control-bus"
     d="M 280 820 C 296 820, 304 820, 320 820"
     id="path1369" />
  <path
     class="control-bus"
     d="M 560 820 C 576 820, 584 820, 600 820"
     id="path1371" />
  <path
     class="fill-bus"
     d="M 900 820 C 916 820, 924 820, 940 820"
     id="path1373" />
  <path
     class="fill-bus"
     d="M 1160 820 C 1176 820, 1184 820, 1200 820"
     id="path1375" />
  <path
     class="fill-bus-tail"
     d="M 1310 796 C 1310 752, 1184 736, 1024 732 C 960 729, 920 724, 920 716"
     id="path1377" />
  <polygon
     class="fill-arrow"
     points="920,696 908,718 932,718"
     id="path1377-arrow" />
</svg>
'''
(ASSETS_DIR / "cache4way_datapath.svg").write_text(DATAPATH_SVG, encoding="utf-8")

print(f"Arquivos prontos em {REPO}")
print(f"Fontes Verilog: {len(list(SRC_DIR.glob('*.v')))}")


In [3]:
#@title 3. Executar simulação
# Compila os arquivos criados na célula anterior e executa o testbench auto-verificável.
import subprocess

%cd /content/Simulador-Cache-4-Way/src
#!make sim
subprocess.run(["make", "sim"], check=True)
print("Simulação executada com sucesso.")


/content/Simulador-Cache-4-Way/src
Simulação executada com sucesso.


## Visualização do datapath

O SVG autoral do datapath é editável no Inkscape e contém marcadores `@...` que são substituídos com os dados gerados pela simulação.


In [4]:
#@title 4. Exibir desenho estático do datapath
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/Simulador-Cache-4-Way")
SVG_PATH = REPO / "assets" / "cache4way_datapath.svg"

def display_responsive_svg(svg_text):
    display(HTML(f"""
    <div style="width:100%; overflow-x:auto; padding: 8px 0;">
      <div style="min-width: 1120px; max-width: 1600px; margin: 0 auto;">
        {svg_text.replace('<svg', '<svg style="width:100%; height:auto; display:block;"', 1)}
      </div>
    </div>
    """))

display_responsive_svg(SVG_PATH.read_text(encoding="utf-8"))


## Visualizações dinâmicas sincronizadas

O explorador abaixo controla duas leituras complementares do mesmo acesso:

- **Datapath:** mostra os sinais locais do acesso atual, o caminho de decisão e a saída.
- **Matriz de estado:** mostra o estado global da cache inteira após o acesso, em `4 conjuntos x 4 vias`.

Use o seletor `Roteiro` para alternar entre a validação técnica e a demonstração global. Depois use `Anterior`, `Próximo` ou o seletor de passo para navegar pela sequência escolhida.


In [5]:
#@title 5. Preparar visualizações interativas
import csv
import html
import re
import subprocess
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

REPO = Path("/content/Simulador-Cache-4-Way")
SRC_DIR = REPO / "src"
SVG_PATH = REPO / "assets" / "cache4way_datapath.svg"
VALIDATION_TRACE_PATH = SRC_DIR / "trace.csv"
VALIDATION_GRID_PATH = SRC_DIR / "trace_grid.csv"
DEMO_TRACE_PATH = SRC_DIR / "trace_demo.csv"
DEMO_GRID_PATH = SRC_DIR / "trace_grid_demo.csv"
TRACE_PATHS = (
    VALIDATION_TRACE_PATH,
    VALIDATION_GRID_PATH,
    DEMO_TRACE_PATH,
    DEMO_GRID_PATH,
)

def ensure_trace_files():
    missing = [path for path in TRACE_PATHS if not path.exists()]
    if not missing:
        return
    if not SRC_DIR.exists():
        raise FileNotFoundError(
            "Arquivos do projeto ainda não criados em /content. Execute primeiro a célula 2."
        )

    print("Traces ausentes. Executando novamente `make sim` para gerá-los...")
    subprocess.run(["make", "sim"], cwd=SRC_DIR, check=True)

    missing = [path for path in TRACE_PATHS if not path.exists()]
    if missing:
        missing_names = ", ".join(path.name for path in missing)
        raise FileNotFoundError(
            "Os traces ainda não foram gerados após `make sim`: "
            f"{missing_names}. A preparação local provavelmente não foi executada; "
            "execute novamente a célula 2 e depois a célula 3."
        )

ensure_trace_files()

def load_trace_pair(trace_path, grid_path):
    trace_rows = list(csv.DictReader(trace_path.open(encoding="utf-8")))
    grid_rows = list(csv.DictReader(grid_path.open(encoding="utf-8")))
    grid_rows_by_step = defaultdict(list)
    for grid_row in grid_rows:
        grid_rows_by_step[grid_row["step"]].append(grid_row)
    return trace_rows, grid_rows_by_step

template_svg = SVG_PATH.read_text(encoding="utf-8")
trace_sets = {
    "Validação técnica": load_trace_pair(VALIDATION_TRACE_PATH, VALIDATION_GRID_PATH),
    "Demonstração global": load_trace_pair(DEMO_TRACE_PATH, DEMO_GRID_PATH),
}
rows, grid_rows_by_step = trace_sets["Validação técnica"]

STATE_NAMES = {
    "0": "COMPARE",
    "1": "HIT",
    "2": "MISS",
    "3": "FILL_BLOCK",
    "4": "UPDATE_TAG",
}

WAY_COLORS = {
    "0": ("#4cc9f0", "rgba(14,165,233,.16)"),
    "1": ("#d8b4fe", "rgba(168,85,247,.16)"),
    "2": ("#fde047", "rgba(234,179,8,.16)"),
    "3": ("#86efac", "rgba(34,197,94,.16)"),
}

def event_class(event):
    if event == "hit" or event.startswith("hit-"):
        return "hit"
    if "replace" in event:
        return "replace"
    return "miss"

def responsive_svg_html(svg_text):
    svg_text = svg_text.replace('<svg', '<svg style="width:100%; height:auto; display:block;"', 1)
    return f"""
    <div style="width:100%; overflow-x:auto; padding: 8px 0 4px;">
      <div style="min-width: 1120px; max-width: 1600px; margin: 0 auto;">
        {svg_text}
      </div>
    </div>
    """

def render_svg(row):
    addr = int(row["addr"])
    tag = int(row["tag"])
    line = int(row["line"])
    blk = int(row["blk"])
    replacements = {
        "@addr": row["addr"],
        "@addr_bin": f"{addr:012b}",
        "@tag": row["tag"],
        "@tag_bin": f"{tag:08b}",
        "@line": row["line"],
        "@line_bin": f"{line:02b}",
        "@blk": row["blk"],
        "@blk_bin": f"{blk:02b}",
        "@hit": row["hit"],
        "@selected_way": row["selected_way"],
        "@dout": row["dout"],
        "@state": STATE_NAMES.get(row["state"], row["state"]),
        "@event": row["event"],
    }
    for i in range(4):
        replacements[f"@valid{i}"] = row[f"valid{i}"]
        replacements[f"@tag{i}"] = row[f"tag{i}"]
        replacements[f"@lru{i}"] = row[f"lru{i}"]

    svg = template_svg
    for marker, value in sorted(replacements.items(), key=lambda item: len(item[0]), reverse=True):
        svg = svg.replace(marker, html.escape(str(value)))

    def add_svg_class(svg_text, element_id, class_name):
        pattern = rf'<[^>]*\bid="{re.escape(element_id)}"[^>]*>'
        def append_class(match):
            opening_tag = match.group(0)
            if 'class="' not in opening_tag:
                raise ValueError(f"Elemento SVG sem classe: {element_id}")
            return re.sub(
                r'class="([^"]*)"',
                lambda class_match: f'class="{class_match.group(1)} {class_name}"',
                opening_tag,
                count=1,
            )
        updated_svg, count = re.subn(pattern, append_class, svg_text, count=1)
        if count != 1:
            raise ValueError(f"Elemento SVG não encontrado: {element_id}")
        return updated_svg

    selected = row["selected_way"]
    svg = add_svg_class(svg, f"way{selected}-card", "selected")
    svg = add_svg_class(svg, f"way{selected}-data-path", "selected")
    svg = add_svg_class(svg, "event-badge", event_class(row["event"]))
    remaining = sorted(set(re.findall(r"@\w+", svg)))
    if remaining:
        raise ValueError(f"Marcadores sem substituição: {remaining}")
    return svg

def row_log(row):
    state = STATE_NAMES.get(row["state"], row["state"])
    cells = [
        ("Roteiro", trace_selector.value),
        ("Passo", row["step"]),
        ("Cenário", row["label"]),
        ("Evento", row["event"]),
        ("Endereço", row["addr"]),
        ("Tag / Line / Blk", f'{row["tag"]} / {row["line"]} / {row["blk"]}'),
        ("Hit", row["hit"]),
        ("Via", row["selected_way"]),
        ("Dout", row["dout"]),
        ("FSM", state),
        ("LRU", f'{row["lru0"]}, {row["lru1"]}, {row["lru2"]}, {row["lru3"]}'),
    ]
    cards = "".join(
        f"""<div style="background:rgba(255,255,255,.82); border:1px solid #dbe4ee;
                    border-radius:14px; padding:10px 12px; box-shadow:0 8px 20px rgba(15,23,42,.08);
                    min-width:0; height:58px; overflow:hidden; box-sizing:border-box;">
              <div style="font:600 11px Georgia,serif; color:#64748b; height:14px; line-height:14px;">{html.escape(name)}</div>
              <div style="font:800 15px ui-monospace,Menlo,Consolas,monospace; color:#0f172a;
                          line-height:18px; max-height:36px; white-space:normal; overflow:hidden;
                          overflow-wrap:break-word; word-break:normal;">{html.escape(str(value))}</div>
            </div>"""
        for name, value in cells
    )
    return f"""
    <div style="max-width:1600px; margin:10px auto 0; padding:14px; border-radius:20px;
                background:linear-gradient(135deg, rgba(248,251,255,.92), rgba(239,246,255,.72));
                border:1px solid #dbe4ee; box-shadow:0 12px 28px rgba(15,23,42,.10);">
      <div style="font:800 16px Georgia,serif; color:#1e293b; margin-bottom:10px;">Log do acesso selecionado</div>
      <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:10px;">{cards}</div>
    </div>
    """

def cache_grid_html(row, snapshot):
    active_set = row["line"]
    active_way = row["selected_way"]
    by_position = {(cell["set"], cell["way"]): cell for cell in snapshot}
    way_headers = "".join(
        f'<div class="cg-head cg-way{way}">Via {way}</div>'
        for way in range(4)
    )
    body_rows = []
    for set_idx in range(4):
        row_active = str(set_idx) == active_set
        cells = [f'<div class="cg-set {"active" if row_active else ""}">Conj. {set_idx}</div>']
        for way_idx in range(4):
            cell = by_position[(str(set_idx), str(way_idx))]
            valid = cell["valid"] == "1"
            selected = row_active and str(way_idx) == active_way
            tag = f'0x{int(cell["tag"]):02X}'
            lru = cell["lru"] if valid else "-"
            color, tint = WAY_COLORS[str(way_idx)]
            classes = ["cg-card", f"way{way_idx}"]
            if not valid:
                classes.append("invalid")
            if selected:
                classes.append("selected")
            cells.append(
                f"""<div class="{' '.join(classes)}" style="--way-color:{color}; --way-tint:{tint};">
                      <div class="cg-tag">Tag: {tag}</div>
                      <div class="cg-badges">
                        <span class="cg-valid {'on' if valid else 'off'}">V: {cell["valid"]}</span>
                        <span class="cg-lru">LRU: {lru}</span>
                      </div>
                    </div>"""
            )
        body_rows.append('<div class="cg-row">' + ''.join(cells) + '</div>')

    return f"""
    <style>
      .cg-shell {{
        max-width:1600px; margin:18px auto 0; padding:24px; border-radius:24px;
        background:radial-gradient(circle at top left, rgba(14,165,233,.12), transparent 34%),
                   linear-gradient(145deg, #07111f, #081526 52%, #0a1830);
        border:1px solid rgba(96,165,250,.26); color:#cbd5e1;
        box-shadow:0 18px 40px rgba(2,6,23,.38), inset 0 1px 0 rgba(148,163,184,.14);
      }}
      .cg-top {{ display:flex; justify-content:space-between; gap:20px; align-items:flex-start; margin-bottom:18px; }}
      .cg-title {{ font:800 24px ui-monospace,Menlo,Consolas,monospace; color:#7dd3fc; letter-spacing:.03em; }}
      .cg-subtitle {{ margin-top:6px; color:#94a3b8; font:500 13px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-config {{ min-width:250px; padding:14px 16px; border-radius:16px; background:rgba(15,23,42,.74);
                    border:1px solid rgba(125,211,252,.22); font:600 12px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-config b {{ color:#7dd3fc; }}
      .cg-grid {{ overflow-x:auto; }}
      .cg-header, .cg-row {{ min-width:860px; display:grid; grid-template-columns:140px repeat(4, 1fr); gap:10px; }}
      .cg-header {{ margin-bottom:10px; }}
      .cg-head {{ padding:12px 14px; border-bottom:1px solid rgba(148,163,184,.24);
                  font:800 14px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-way0 {{ color:#4cc9f0; }} .cg-way1 {{ color:#d8b4fe; }}
      .cg-way2 {{ color:#fde047; }} .cg-way3 {{ color:#86efac; }}
      .cg-row {{ margin-bottom:10px; align-items:stretch; }}
      .cg-set {{ display:flex; align-items:center; padding:0 14px; border-radius:14px;
                 background:rgba(15,23,42,.42); color:#93c5fd; font:800 14px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-set.active {{ color:#e0f2fe; border:1px solid rgba(125,211,252,.55); box-shadow:inset 0 0 0 1px rgba(125,211,252,.12); }}
      .cg-card {{ min-height:74px; padding:12px 14px; border-radius:16px; border:1px solid var(--way-color);
                  background:linear-gradient(145deg, var(--way-tint), rgba(15,23,42,.72));
                  box-shadow:inset 0 1px 0 rgba(255,255,255,.07); }}
      .cg-card.invalid {{ border-color:rgba(148,163,184,.34); background:rgba(15,23,42,.42); filter:saturate(.42); }}
      .cg-card.selected {{ box-shadow:0 0 0 2px rgba(125,211,252,.92), 0 0 28px rgba(56,189,248,.26), inset 0 1px 0 rgba(255,255,255,.10); }}
      .cg-tag {{ color:#e2e8f0; font:800 15px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-badges {{ display:flex; gap:8px; margin-top:10px; }}
      .cg-valid, .cg-lru {{ padding:4px 9px; border-radius:8px; font:800 12px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-valid.on {{ color:#86efac; background:rgba(22,101,52,.34); border:1px solid rgba(74,222,128,.28); }}
      .cg-valid.off {{ color:#fca5a5; background:rgba(127,29,29,.30); border:1px solid rgba(248,113,113,.24); }}
      .cg-lru {{ color:#bae6fd; background:rgba(14,116,144,.22); border:1px solid rgba(56,189,248,.22); }}
      .cg-footer {{ display:flex; gap:12px; flex-wrap:wrap; align-items:center; margin-top:16px; color:#94a3b8;
                    font:600 12px ui-monospace,Menlo,Consolas,monospace; }}
      .cg-pill {{ padding:7px 10px; border-radius:999px; border:1px solid rgba(148,163,184,.20); background:rgba(15,23,42,.54); }}
    </style>
    <section class="cg-shell">
      <div class="cg-top">
        <div>
          <div class="cg-title">Matriz de Estado da Cache (Grid View)</div>
          <div class="cg-subtitle">Estado global após o passo {html.escape(row["step"])}: {html.escape(row["label"])}</div>
        </div>
        <div class="cg-config">
          Configuração: <b>Cache 4-Way</b><br>
          Conjuntos (Sets): <b>4</b><br>
          Vias (Ways): <b>4</b><br>
          Política: <b>LRU</b>
        </div>
      </div>
      <div class="cg-grid">
        <div class="cg-header"><div class="cg-head">Conjunto</div>{way_headers}</div>
        {''.join(body_rows)}
      </div>
      <div class="cg-footer">
        <span class="cg-pill">V:1 = válido</span>
        <span class="cg-pill">V:0 = inválido</span>
        <span class="cg-pill">LRU 0 = via mais recente</span>
        <span class="cg-pill">LRU 3 = via menos recente</span>
        <span class="cg-pill">Conjunto ativo: {html.escape(active_set)}</span>
        <span class="cg-pill">Via selecionada: {html.escape(active_way)}</span>
      </div>
    </section>
    """

trace_selector = widgets.Dropdown(
    options=list(trace_sets.keys()),
    value="Validação técnica",
    description="Roteiro:",
    layout=widgets.Layout(width="280px"),
)
slider = widgets.IntSlider(
    value=1,
    min=1,
    max=len(rows),
    step=1,
    description="Passo:",
    layout=widgets.Layout(width="420px"),
)
svg_output = widgets.Output()
log_output = widgets.Output()
grid_output = widgets.Output()

def show_step(step):
    row = rows[step - 1]
    snapshot = grid_rows_by_step[row["step"]]
    with svg_output:
        clear_output(wait=True)
        display(HTML(responsive_svg_html(render_svg(row))))
    with log_output:
        clear_output(wait=True)
        display(HTML(row_log(row)))
    with grid_output:
        clear_output(wait=True)
        display(HTML(cache_grid_html(row, snapshot)))

def on_slider_change(change):
    if change["name"] == "value":
        show_step(change["new"])

def on_trace_change(change):
    global rows, grid_rows_by_step
    if change["name"] == "value":
        rows, grid_rows_by_step = trace_sets[change["new"]]
        slider.max = len(rows)
        if slider.value != 1:
            slider.value = 1
        else:
            show_step(1)

def previous(_):
    slider.value = max(slider.min, slider.value - 1)

def next_(_):
    slider.value = min(slider.max, slider.value + 1)

def navigation_button(description, color):
    button = widgets.Button(
        description=description,
        layout=widgets.Layout(width="128px"),
    )
    button.style.button_color = color
    return button

def navigation_bar(include_slider=False):
    prev_button = navigation_button("← Anterior", "#e2e8f0")
    next_button = navigation_button("Próximo →", "#93c5fd")
    prev_button.on_click(previous)
    next_button.on_click(next_)
    children = [prev_button, next_button]
    if include_slider:
        children = [trace_selector] + children + [slider]
    return widgets.HBox(
        children,
        layout=widgets.Layout(margin="10px 0 6px", gap="8px"),
    )

slider.observe(on_slider_change, names="value")
trace_selector.observe(on_trace_change, names="value")

controls = navigation_bar(include_slider=True)
datapath_controls = navigation_bar()
log_controls = navigation_bar()
grid_controls = navigation_bar()


### Controles do explorador

O roteiro e o passo selecionados atualizam as três visualizações abaixo.


In [6]:
#@title 6. Exibir controles do explorador
display(controls)
show_step(slider.value)


### Datapath do acesso selecionado


In [7]:
#@title 7. Exibir datapath dinâmico
display(svg_output)
display(datapath_controls)


Output()

### Log do acesso selecionado


In [8]:
#@title 8. Exibir log do acesso
display(log_output)
display(log_controls)


Output()

### Matriz global da cache


In [9]:
#@title 9. Exibir matriz global da cache
display(grid_output)
display(grid_controls)


Output()

## Como interpretar a saída

O testbench imprime cada acesso à cache, valida os sinais esperados e gera dois roteiros de visualização:

- `Validação técnica`: prova os cenários principais do testbench com uma sequência curta.
- `Demonstração global`: preenche todos os conjuntos e mostra hits, LRU, substituições e offsets ao longo da cache inteira.

Arquivos gerados:

- `trace.csv` e `trace_grid.csv`: alimentam o roteiro de validação técnica.
- `trace_demo.csv` e `trace_grid_demo.csv`: alimentam o roteiro de demonstração global.

Sinais do terminal:

- `ACCESS`: mostra o cenário testado, endereço, tag, linha, bloco, hit, via selecionada, dado e LRU.
- `PASS`: indica que o valor observado bateu com o valor esperado.
- `FAIL`: indica erro; o testbench chama `$fatal(1)` e a simulação termina com falha.
- `ALL TESTS PASSED`: indica que todos os cenários principais passaram.

### Campos do log

- `Roteiro`: conjunto de acessos selecionado no explorador.
- `Passo`: posição do acesso na sequência simulada.
- `Cenário`: nome do caso exercitado pelo testbench.
- `Evento`: classe do acesso, como `hit`, `miss-fill` ou `miss-replace`.
- `Endereço`: endereço solicitado à cache.
- `Tag / Line / Blk`: decomposição do endereço em tag, conjunto e offset de bloco.
- `Hit`: indica se o bloco já estava presente na cache.
- `Via`: via usada para o hit ou escolhida para preenchimento/substituição.
- `Dout`: dado devolvido pela cache.
- `FSM`: estado final observado na máquina de estados.
- `LRU`: idades das quatro vias, em que `0` é a mais recente e `3` a menos recente.

A leitura das duas visualizações é complementar: o datapath mostra **como** um acesso é processado; a matriz mostra **como a cache inteira ficou** depois desse acesso.
